# Síntesis Integral y Ejecutable de la Tesis · Operación y Dimensionamiento de PV + BESS + Autogeneración Térmica (V1 y V2)

Este cuaderno reúne de manera organizada, pedagógica y **completamente ejecutable** los métodos, las simulaciones y los resultados centrales de toda la tesis. Su estructura recorre las **tres etapas del estudio**, combinando **corridas anuales completas en vivo** de las políticas operativas con el análisis de los barridos factoriales, los benchmarks de programación entera mixta (MILP) en PyPSA y las evaluaciones financieras multianuales (20 años).

---

## Mapa de navegación del cuaderno

| Etapa | Sección | Contenido principal y qué ejecuta en vivo |
|---|---|---|
| **Preliminar** | **§1 · Entorno y parámetros maestros** | Activa la aceleración Numba (`.venv/numba`), verifica las fuentes de datos y presenta los supuestos técnicos, económicos, tarifarios, térmicos y ambientales vigentes. |
| **Etapa I**<br>*(PV + BESS + Red,*<br>*sin térmica)* | **§2 · Alcance industrial y simulación en vivo (K3)** | Explica la delimitación $E_{\text{PV}}^{\text{anual}} \le E_{\text{demanda}}^{\text{anual}}$, define las 5 reglas (`A`, `B`, `C`, `D`, `F`) y **simula en vivo los 20 años completos (8.760 h/año)** en Oriente / K3 bajo `T0` y `T1`, comprobando su coincidencia exacta con el cierre oficial. |
| | **§3 · Factorial en 5 sitios y descomposición** | Analiza los 200 casos + 30 líneas base en las 5 regiones UPME (`K1–K4`), descompone determinísticamente la varianza y grafica el aporte de cada factor en el diseño completo y en el subconjunto orientado al usuario (`A/B/D`). |
| | **§4 · Benchmarks `OPT_B` y `OPT_FULL`** | Separa algebraicamente la pérdida de las heurísticas en **brecha de decisión** (previsión perfecta, $0,35\text{–}1,43\,\%$) y **brecha de flexibilidad** (ampliación de acciones operativas, $2,05\text{–}3,79\,\%$). |
| **Etapa II**<br>*(Integración con*<br>*flota térmica G1–G4)* | **§5 · Compromiso térmico y simulación anual en vivo** | Formula el gestor térmico diario por programación dinámica ($6^4 = 1.296$ estados) y **ejecuta en vivo las trayectorias anuales de 8.760 h** de `RED_TERMICA`, `PV_SIN_BESS`, `A_INT`, `B_INT` y `D_INT` (más ventana demostrativa de `C_INT`/`F_INT`), validando balances y cierres cíclicos. |
| | **§6 · Comparación conjunta A/B/D/C/F y cotas MILP** | Contrasta las 5 políticas integradas frente a los intervalos de cota de sus benchmarks compatibles (`OPT_AB`, `OPT_D`, `OPT_CF`), desglosando costos, arranques térmicos, rutas de energía y perfiles horarios semanales. |
| | **§7 · Factorial térmico `K1–K4` y VPN de K3 (20 años)** | Explica por qué el ganador cambia entre `A_INT` (`K1/K3`) y `D_INT` (`K2/K4`) y presenta el VPN total e incremental del BESS a 20 años en K3 (362 despachos anuales). |
| **Etapa III**<br>*(Extensión V2:*<br>*disponibilidad, carbono*<br>*y dimensionamiento)* | **§8 · Disponibilidad V2, carbono y piloto K3 en vivo** | Incorpora el calendario horario de disponibilidad térmica (95,01 %) y el precio interno de carbono ($p_C$), y **ejecuta en vivo las corridas anuales de 8.760 h** en `V2_p0` (0 USD/t) y `V2_p50` (50 USD/t) bajo `T0` y `T1`. |
| | **§9 · Diagnóstico de apagado anticipado en `B_INT`** | Analiza las situaciones `N1–N4` de unidades térmicas heredadas encendidas en su mínimo técnico (G2 a 2,4 MW) y **ejecuta en vivo** el contraste de las alternativas `ALT1` y `ALT2` en la ventana representativa. |
| | **§10 · Malla `M01–M12` y benchmarks finalistas** | Sintetiza las 6.400 anualidades y los 600 VPN de las 12 configuraciones (`M01–M12`), presenta las tres comparaciones separadas (reglas a tamaño fijo, mejor tamaño por regla y ranking entre configuraciones) y contrasta los 17 benchmarks anuales de los finalistas. |
| | **§11 · Sensibilidad de CAPEX del BESS (`H_B4` vs `H_B3`)** | **Recalcula en vivo** los CAPEX, OPEX y VPN de las 12 configuraciones bajo los segmentos de costo local `B4` y `B3`, verificando la estabilidad de `M01` y los cambios de tamaño preferido por regla bajo `T1`. |
| **Cierre** | **§12 · Conclusiones integradas y trazabilidad** | Cuadro sinóptico de hallazgos por objetivo, limitaciones metodológicas declaradas y verificación automática de integridad SHA-256 de todas las fuentes del repositorio. |

> **Garantía de no sobrescritura:** todas las salidas generadas por este cuaderno se escriben exclusivamente en `resultados_nb15/` y `figuras_nb15/`. Los archivos oficiales de `resultados_nb08/`, `resultados_correccion_numerica/20260920/` y `resultados_carbono_v2/` se leen en modo de solo lectura y sus huellas SHA-256 se verifican al inicio y al final.

## 1 · Configuración del entorno, aceleración Numba y parámetros maestros

El motor de programación dinámica térmica (`evaluador_dia` en `termica_politicas.py`) resuelve exactamente el compromiso horario de las cuatro unidades a gas sobre $6^4 = 1.296$ estados por hora. Al activar el compilador JIT de Numba desde `.venv/numba` (`termica_dp_acelerada.activar()`), se acelera la evaluación horaria conservando las reglas y las comprobaciones numéricas del motor.

In [1]:
import sys, time, json, math, hashlib, warnings, logging
from decimal import Decimal, ROUND_HALF_UP, ROUND_FLOOR, ROUND_CEILING
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

ROOT = Path.cwd()
if (ROOT / ".venv" / "numba").exists():
    sys.path.insert(0, str(ROOT / ".venv" / "numba"))
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / ".build"))

warnings.filterwarnings("ignore")
for _logger_name in ("pypsa", "linopy"):
    logging.getLogger(_logger_name).setLevel(logging.ERROR)

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import FuncFormatter
from matplotlib.lines import Line2D
from IPython.display import Markdown, display

# Activación del acelerador Numba para programación dinámica térmica
try:
    from termica_dp_acelerada import activar as activar_numba
    NUMBA_ACTIVO = bool(activar_numba())
except Exception:
    NUMBA_ACTIVO = False

import costos
import costos_curvas as CC
import termica
import carbono as C
import disponibilidad_termica as DT

OUT = ROOT / "resultados_nb15"
FIG = ROOT / "figuras_nb15"
OUT.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)

# Paleta institucional validada para daltonismo (.build/validar_paleta.py)
PALETA = {
    "A": "#2a78d6", "A_INT": "#2a78d6", "A Autoconsumo": "#2a78d6",
    "B": "#eb6834", "B_INT": "#eb6834", "B Merito diario": "#eb6834",
    "D": "#1baf7a", "D_INT": "#1baf7a", "D Merito+Export": "#1baf7a",
    "C": "#a03060", "C_INT": "#a03060", "C Merchant": "#a03060",
    "F": "#0d5f4f", "F_INT": "#0d5f4f", "F Merchant reserva": "#0d5f4f",
    "OPT_B": "#8b8a83", "OPT_FULL": "#52514e",
    "OPT_AB": "#52514e", "OPT_D": "#52514e", "OPT_CF": "#52514e",
    "SIN_BESS": "#8b8a83", "PV_SIN_BESS": "#8b8a83", "RED_TERMICA": "#52514e",
}

plt.rcParams.update({
    "figure.dpi": 115,
    "savefig.dpi": 200,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": "#d8d8d4",
    "grid.linewidth": 0.6,
    "axes.edgecolor": "#d8d8d4",
    "font.size": 9.5,
})

def es(x, dec=2):
    # Formatea números con punto de miles y coma decimal (convención en español).
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return "—"
    return f"{float(x):,.{dec}f}".replace(",", "~").replace(".", ",").replace("~", ".")

def intervalo_es(lo, hi, dec=2):
    # Formatea intervalos [lo; hi] redondeando hacia afuera para preservar la cota.
    u = Decimal(10) ** (-dec)
    a = Decimal(str(lo)).quantize(u, rounding=ROUND_FLOOR)
    b = Decimal(str(hi)).quantize(u, rounding=ROUND_CEILING)
    s_a = f"{a:,.{dec}f}".replace(",", "~").replace(".", ",").replace("~", ".")
    s_b = f"{b:,.{dec}f}".replace(",", "~").replace(".", ",").replace("~", ".")
    return s_a if a == b else f"[{s_a}; {s_b}]"

def mostrar_tabla(df, dec=2):
    # Muestra un DataFrame como tabla Markdown limpia con formato en español.
    copia = df.copy()
    for col in copia.columns:
        if pd.api.types.is_float_dtype(copia[col]):
            copia[col] = copia[col].map(lambda v: es(v, dec))
    encabezado = "| " + " | ".join(map(str, copia.columns)) + " |"
    separador = "|" + "|".join("---:" if pd.api.types.is_numeric_dtype(df[c]) else "---" for c in df.columns) + "|"
    filas = ["| " + " | ".join(map(str, r)) + " |" for r in copia.to_numpy()]
    display(Markdown("\n".join([encabezado, separador] + filas)))

def guardar_fig(fig, nombre):
    # Guarda cada figura en PNG y SVG en figuras_nb15/ y la muestra en el cuaderno.
    fig.savefig(FIG / f"{nombre}.png", bbox_inches="tight")
    fig.savefig(FIG / f"{nombre}.svg", bbox_inches="tight")
    plt.show()
    plt.close(fig)

# Registro de huellas SHA-256 de archivos clave para auditar que nada se modifica
FUENTES_CLAVE = [
    ROOT / "resultados_nb08/factorial_operativo_TRM_actual.csv",
    ROOT / "resultados_nb08/benchmarks_operativos_TRM_actual.csv",
    ROOT / "resultados_nb08/gaps_benchmark_TRM_actual.csv",
    ROOT / "resultados_correccion_numerica/20260920/notebook14/14_comparacion_conjunta.csv",
    ROOT / "resultados_correccion_numerica/20260920/factorial/resumen.csv",
    ROOT / "resultados_correccion_numerica/20260920/vpn/vpn_resumen.csv",
    ROOT / "resultados_carbono_v2/piloto_K3_20260926_tablas_v2/resumen_anual.csv",
    ROOT / "resultados_carbono_v2/vpn_K3_20260926/vpn.csv",
    ROOT / "resultados_carbono_v2/malla_20260926/resultados_por_configuracion.csv",
    ROOT / "resultados_carbono_v2/sensibilidad_capex_bess_20260927/capex_opex_por_configuracion.csv",
]
HASHES_INICIALES = {p.as_posix(): hashlib.sha256(p.read_bytes()).hexdigest() for p in FUENTES_CLAVE}

supuestos_df = pd.DataFrame([
    ("Horizonte y resolución", "8.760 horas/año (año 2025) · Evaluación financiera a 20 años", "Series horarias reales + TMY NSRDB"),
    ("Tasa de cambio (TRM) y descuento", f"{es(costos.TRM_COP_USD, 2)} COP/USD · WACC real = {es(costos.WACC*100, 1)} %", f"TRM vigente al {costos.TRM_FECHA_REFERENCIA} (costos.py)"),
    ("Señal T0 (cuasi-plana N3 corregida)", "Punta (06:00–21:00): 659,9496 COP/kWh · Fuera de punta: 653,2247 COP/kWh", "CORRECCION_TARIFAS.md (13/09/2026)"),
    ("Señal T1 (horaria indexada)", "Indexada al precio de bolsa nacional 2025 recalibrada a media anual de T0", "02_Optimizacion_portafolio_PV_BESS.ipynb"),
    ("BESS técnico y vida útil", "DoD = 80 % · RTE = 90 % (η_c = η_d = √0,90) · 7.200 EFC · 20 años calendario", "motor_multianual.py (sin pérdida progresiva)"),
    ("Configuración representativa K3 (M01)", "PV = 114,415 MWp DC · BESS = 45 MW / 280 MWh (224 MWh útiles)", "Equivalencia energética anual f_PV = 1,0"),
    ("Flota térmica G1–G4 (Etapas II y III)", "4 unidades a gas · P disponibles: 8,0 / 8,0 / 6,0 / 3,2 MW (25,2 MW en total) · Pmin: 2,40 / 2,40 / 1,80 / 0,96 MW", "termica.py (máx. 1 arranque y 1 parada/día)"),
    ("Combustible, CVOM y arranque", "Gas = 20 USD/MMBtu · HR = 9,638 / 9,638 / 11,089 / 12,240 MMBtu/MWh · CVOM = 9,28 / 15,32 USD/MWh · Arranque = 100 USD", "CAMBIO_CVOM.md (18/09/2026)"),
    ("Disponibilidad térmica V2 (Etapa III)", "95,01 % disponible (8.323 h) · 437 h indisponibles (101 h demanda baja + 336 h mant. sep.)", "disponibilidad_termica.py (CARBONO_V2.md)"),
    ("Factores de emisión (Etapa III)", "Gas FECOC = 0,058597 t CO₂/MMBtu · Red XM 2025 = 0,097018 t CO₂e/MWh (sens. UPME 2024 = 0,220)", "costos.py / CARBONO_V2.md"),
], columns=["Bloque", "Parámetro / Valor vigente", "Fuente / Referencia"])

print(f"Entorno inicializado correctamente. Aceleración Numba activa: {NUMBA_ACTIVO}")
print(f"Archivos fuente verificados: {len(HASHES_INICIALES)} tablas maestras protegidas por SHA-256.")
mostrar_tabla(supuestos_df)

Matplotlib is building the font cache; this may take a moment.


Entorno inicializado correctamente. Aceleración Numba activa: True
Archivos fuente verificados: 10 tablas maestras protegidas por SHA-256.


| Bloque | Parámetro / Valor vigente | Fuente / Referencia |
|---|---|---|
| Horizonte y resolución | 8.760 horas/año (año 2025) · Evaluación financiera a 20 años | Series horarias reales + TMY NSRDB |
| Tasa de cambio (TRM) y descuento | 3.126,08 COP/USD · WACC real = 10,0 % | TRM vigente al 2026-09-05 (costos.py) |
| Señal T0 (cuasi-plana N3 corregida) | Punta (06:00–21:00): 659,9496 COP/kWh · Fuera de punta: 653,2247 COP/kWh | CORRECCION_TARIFAS.md (13/09/2026) |
| Señal T1 (horaria indexada) | Indexada al precio de bolsa nacional 2025 recalibrada a media anual de T0 | 02_Optimizacion_portafolio_PV_BESS.ipynb |
| BESS técnico y vida útil | DoD = 80 % · RTE = 90 % (η_c = η_d = √0,90) · 7.200 EFC · 20 años calendario | motor_multianual.py (sin pérdida progresiva) |
| Configuración representativa K3 (M01) | PV = 114,415 MWp DC · BESS = 45 MW / 280 MWh (224 MWh útiles) | Equivalencia energética anual f_PV = 1,0 |
| Flota térmica G1–G4 (Etapas II y III) | 4 unidades a gas · P disponibles: 8,0 / 8,0 / 6,0 / 3,2 MW (25,2 MW en total) · Pmin: 2,40 / 2,40 / 1,80 / 0,96 MW | termica.py (máx. 1 arranque y 1 parada/día) |
| Combustible, CVOM y arranque | Gas = 20 USD/MMBtu · HR = 9,638 / 9,638 / 11,089 / 12,240 MMBtu/MWh · CVOM = 9,28 / 15,32 USD/MWh · Arranque = 100 USD | CAMBIO_CVOM.md (18/09/2026) |
| Disponibilidad térmica V2 (Etapa III) | 95,01 % disponible (8.323 h) · 437 h indisponibles (101 h demanda baja + 336 h mant. sep.) | disponibilidad_termica.py (CARBONO_V2.md) |
| Factores de emisión (Etapa III) | Gas FECOC = 0,058597 t CO₂/MMBtu · Red XM 2025 = 0,097018 t CO₂e/MWh (sens. UPME 2024 = 0,220) | costos.py / CARBONO_V2.md |

---

# Etapa I · Sistema PV + BESS + Red (Sin Autogeneración Térmica)

## 2 · Delimitación de alcance industrial y simulación multianual en vivo (Oriente / K3)

En la primera etapa se estudia un usuario industrial conectado a la red colombiana que incorpora generación fotovoltaica propia y un sistema de almacenamiento en baterías (BESS), **sin considerar todavía la flota de autogeneración térmica**.

### 2.1 · Condición de alcance industrial y penetración PV normalizada

Para preservar la naturaleza de autogenerador industrial (y no convertir el proyecto en una planta *merchant* de generación pura), se impone que la energía solar anual generada en el año 1 no supere la demanda anual del usuario ($158.742,57\text{ MWh/año}$):

$$f_{\text{PV}} = \frac{E_{\text{PV}}^{\text{anual}}}{E_{\text{demanda}}^{\text{anual}}} \le 1,0 \qquad \Longrightarrow \qquad \text{PV}_{\text{MWp}}(\text{sitio}) = f_{\text{PV}} \cdot \frac{E_{\text{demanda}}^{\text{anual}}}{\text{Yield}_{\text{anual}}(\text{sitio})}$$

### 2.2 · Las cinco reglas operativas evaluadas

| Regla | Nombre corto | Carga del BESS | Descarga del BESS | Propósito operativo |
|---|---|---|---|---|
| **A** | Autoconsumo | Solo excedente PV | Toda demanda residual disponible | Maximizar sustitución física inmediata de compras de red |
| **B** | Mérito diario | Solo excedente PV | Horas de mayor costo tarifario del día | Desplazar energía solar hacia las horas de mayor tarifa de importación |
| **D** | Mérito + Exportación | Solo excedente PV | Demanda cara + exportación a red | Igual a B, pero permite exportar desde el BESS cuando el precio de exportación supera el umbral |
| **C** | *Merchant* (carga red) | Excedente PV + Red barata | Demanda cara + exportación a red | Permite comprar de la red en horas baratas para ciclar la batería |
| **F** | *Merchant* + Reserva solar | Excedente PV + Red (con tope) | Demanda cara + exportación a red | Restringe la compra de red reservando capacidad de SOC para el excedente solar esperado |

### 2.3 · Simulación en vivo de los 20 años completos en Oriente / K3 (`8.760 h × 20 años`)

La celda siguiente carga el motor original del cuaderno 02 (`factorial_operativo.py` + `motor_multianual.py`) y **simula en vivo las 5 reglas operativas más la línea base sin BESS durante los 20 años completos** en la región **Oriente** con la configuración representativa **K3** ($114,415\text{ MWp}$ PV; $45\text{ MW} / 280\text{ MWh}$ BESS) bajo las señales `T0` y `T1`. Además, contrasta el resultado calculado en vivo contra la tabla oficial congelada en `resultados_nb08/factorial_operativo_TRM_actual.csv`.

In [2]:
import factorial_operativo as FO

t0_etapa1 = time.perf_counter()
ns_fo = FO.cargar()
FAC_NB08 = pd.read_csv(ROOT / "resultados_nb08/factorial_operativo_TRM_actual.csv")
BASE_NB08 = pd.read_csv(ROOT / "resultados_nb08/baseline_pv_TRM_actual.csv")

MUSD_FACTOR = costos.TRM_COP_USD * 1e6
E_DEM_ANUAL = float(ns_fo["LOAD"].sum())
YIELD_ORIENTE = float(ns_fo["PU_SITIO"]["Oriente"].sum())
PV_K3_ORIENTE = 1.000 * E_DEM_ANUAL / YIELD_ORIENTE  # 114,415019 MWp

# 1) Simulación en vivo de las líneas base PV-only (20 años) en Oriente para T0 y T1
base_vivo = {}
for senal in ("T0", "T1"):
    res_b = FO.evaluar(ns_fo, "Oriente", senal, ns_fo["ESTRATEGIAS_FACTORIAL"][0], PV_K3_ORIENTE, 0.0, 0.0)
    assert res_b["Fallos"] == "", f"Fallo en línea base {senal}: {res_b['Fallos']}"
    base_vivo[senal] = res_b["VPN_COP"] / MUSD_FACTOR

# 2) Simulación en vivo de las 5 estrategias (20 años = 175.200 horas c/u) en Oriente / K3
filas_vivo_e1 = []
for senal in ("T0", "T1"):
    for est in ns_fo["ESTRATEGIAS_FACTORIAL"]:
        res = FO.evaluar(ns_fo, "Oriente", senal, est, PV_K3_ORIENTE, 45.0, 280.0)
        assert res["Fallos"] == "", f"Fallo de validación en {senal} / {est}: {res['Fallos']}"
        vpn_musd = res["VPN_COP"] / MUSD_FACTOR
        delta_bess_musd = vpn_musd - base_vivo[senal]

        # Verificación contra el CSV oficial congelado en resultados_nb08/
        ref_row = FAC_NB08[
            (FAC_NB08.Sitio == "Oriente") &
            (FAC_NB08.Senal == senal) &
            (FAC_NB08.Estrategia == est) &
            (FAC_NB08.K == "K3")
        ].iloc[0]
        err_vpn = abs(vpn_musd - float(ref_row.VPN_MUSD))
        err_delta = abs(delta_bess_musd - float(ref_row.Delta_VPN_BESS_MUSD))
        assert err_vpn < 1e-6 and err_delta < 1e-6, f"Discrepancia en {senal}/{est}: {err_vpn}"

        filas_vivo_e1.append({
            "Señal": senal,
            "Regla": est.split()[0],
            "Estrategia": est,
            "VPN proyecto [MUSD]": vpn_musd,
            "ΔVPN BESS [MUSD]": delta_bess_musd,
            "TIR [%]": res["TIR"] * 100.0,
            "EFC 20 años": res["EFC_total"],
            "Renovaciones": int(res["N_renovaciones"]),
            "Importación año 1 [GWh]": res["Importacion_MWh"] / 1e3,
            "Carga red año 1 [GWh]": res["Carga_red_MWh"] / 1e3,
            "Error vs CSV [MUSD]": max(err_vpn, err_delta),
        })

dt_etapa1 = time.perf_counter() - t0_etapa1
DF_VIVO_E1 = pd.DataFrame(filas_vivo_e1)
print(f"Simulación multianual en vivo completada en {es(dt_etapa1, 2)} s "
      f"(10 casos × 20 años + 2 líneas base = 2.102.400 pasos horarios validados).")
print(f"Error máximo frente a resultados_nb08/factorial_operativo_TRM_actual.csv: "
      f"{DF_VIVO_E1['Error vs CSV [MUSD]'].max():.2e} MUSD.")
mostrar_tabla(DF_VIVO_E1.drop(columns=["Estrategia", "Error vs CSV [MUSD]"]), dec=3)

,mín [COP/kWh],media [COP/kWh],máx [COP/kWh],rango máx/mín,costo demanda [mil MM COP]
T0 — señal cuasi-plana,653.2,657.4,659.9,1.01x,104.36
T1 — señal horaria indexada,510.2,653.2,"2,636.4",5.17x,104.36


Tarifa,Estrategia,Descarga [MWh],Export BESS [MWh],Carga red [MWh],Ciclos/día,Precio medio descarga,Validación
T0 — señal cuasi-plana,A Autoconsumo,"31,689",0,0,0.90,659.0,OK
T0 — señal cuasi-plana,B Merito diario,"31,672",0,0,0.90,659.2,OK
T0 — señal cuasi-plana,C Merchant,"30,484",300,"5,834",0.87,659.9,OK
T0 — señal cuasi-plana,D Merito+Export,"31,747",180,0,0.91,659.2,OK
T1 — señal horaria indexada,A Autoconsumo,"31,689",0,0,0.90,761.1,OK
T1 — señal horaria indexada,B Merito diario,"31,497",0,0,0.90,769.6,OK
T1 — señal horaria indexada,C Merchant,"32,004",0,"21,083",0.91,770.4,OK
T1 — señal horaria indexada,D Merito+Export,"31,497",0,0,0.90,769.6,OK


Simulación multianual en vivo completada en 9,72 s (10 casos × 20 años + 2 líneas base = 2.102.400 pasos horarios validados).
Error máximo frente a resultados_nb08/factorial_operativo_TRM_actual.csv: 1.78e-14 MUSD.


| Señal | Regla | VPN proyecto [MUSD] | ΔVPN BESS [MUSD] | TIR [%] | EFC 20 años | Renovaciones | Importación año 1 [GWh] | Carga red año 1 [GWh] |
|---|---|---:|---:|---:|---:|---:|---:|---:|
| T0 | A | 91,420 | 29,760 | 19,752 | 5.667,702 | 0 | 30,598 | 0,000 |
| T0 | B | 91,436 | 29,776 | 19,753 | 5.667,551 | 0 | 30,598 | 0,000 |
| T0 | C | 43,689 | -17,970 | 14,814 | 3.129,056 | 0 | 69,799 | 8,035 |
| T0 | D | 90,570 | 28,910 | 19,667 | 5.751,113 | 0 | 31,269 | 0,000 |
| T0 | F | 51,619 | -10,041 | 15,649 | 3.129,056 | 0 | 63,753 | 1,988 |
| T1 | A | 95,389 | 37,513 | 20,141 | 5.667,702 | 0 | 30,598 | 0,000 |
| T1 | B | 96,042 | 38,165 | 20,203 | 5.654,725 | 0 | 30,727 | 0,000 |
| T1 | C | 34,457 | -23,420 | 13,839 | 3.745,182 | 0 | 81,698 | 27,739 |
| T1 | D | 95,388 | 37,512 | 20,138 | 5.736,787 | 0 | 31,257 | 0,000 |
| T1 | F | 61,398 | 3,521 | 16,684 | 3.741,855 | 0 | 59,669 | 5,679 |

## 3 · Factorial operativo en 5 regiones UPME y descomposición determinística de la variación

El experimento factorial completo sin térmica cruza **5 sitios UPME** (Caribe, Suroccidente, Nordeste, Oriente, Antioquia) $\times$ **2 señales** (`T0`, `T1`) $\times$ **5 estrategias** (`A`, `B`, `C`, `D`, `F`) $\times$ **4 configuraciones** (`K1–K4`), totalizando **200 casos con batería** y **30 líneas base sin batería**.

- **Por qué se separa el análisis en dos conjuntos:** las reglas *merchant* `C` y `F` permiten carga desde red en el primer cuartil tarifario. El costo de inversión, las pérdidas de ida y vuelta ($\text{RTE} = 90\,\%$) y el despacho resultante hacen que $\Delta\text{VPN}_{\text{BESS}} < 0$ en **46 de los 200 casos**, todos pertenecientes a `C` o `F`; no todos los casos de esas dos reglas son negativos.
- Por ello, cuando se descomponen las sumas de cuadrados sobre las **5 estrategias**, el factor `Estrategia` absorbe el **$65,73\,\%$** de la varianza total de $\Delta\text{VPN}_{\text{BESS}}$ y su interacción `Estrategia × K` otro **$21,95\,\%$**. En cambio, al restringir el análisis a las **reglas orientadas al usuario (`A`, `B`, `D`)** —que siempre tienen $\Delta\text{VPN}_{\text{BESS}} > 0$ (mínimo $+14,71\text{ MUSD}$)—, la elección de regla explica apenas el **$0,07\,\%$** de la variación y lo que domina es la **configuración `K` ($66,38\,\%$)** seguida de la **señal tarifaria ($28,02\,\%$)** y el **sitio ($4,11\,\%$)**.

In [3]:
EF_NB08 = pd.read_csv(ROOT / "resultados_nb08/efectos_factoriales_TRM_actual.csv")
GAP_ABD_NB08 = pd.read_csv(ROOT / "resultados_nb08/gap_ABD_TRM_actual.csv")

# Tabla de configuraciones K1-K4 y rendimientos por sitio
CONFS_K = ["K1", "K2", "K3", "K4"]
SITIOS_ORD = ["Caribe", "Suroccidente", "Nordeste", "Oriente", "Antioquia"]
ABD_NOMBRES = ["A Autoconsumo", "B Merito diario", "D Merito+Export"]

tabla_k = FAC_NB08.groupby("K").agg(
    f_PV=("f_PV", "first"),
    P_BESS_MW=("P_BESS_MW", "first"),
    E_BESS_MWh=("E_BESS_MWh", "first"),
    Duracion_h=("Duracion_h", "first"),
    r_medio=("r_medio", "mean"),
    Delta_VPN_medio_ABD_MUSD=("Delta_VPN_BESS_MUSD", lambda s: s[FAC_NB08.loc[s.index, "Estrategia"].isin(ABD_NOMBRES)].mean()),
).loc[CONFS_K].reset_index()
tabla_k["Régimen"] = [
    "Almacenamiento abundante",
    "Intermedio",
    "Configuración representativa",
    "Almacenamiento escaso",
]
print("Resumen de las 4 configuraciones categóricas K1–K4:")
mostrar_tabla(tabla_k, dec=3)

# Descomposición de varianza (ANOVA determinístico sobre Delta_VPN_BESS)
ef_delta = EF_NB08[
    EF_NB08.Variable.isin(["Delta_VPN_BESS_MUSD", "Delta_VPN_BESS_MUSD_ABD"])
].copy()
piv_ef = ef_delta.pivot(index="Termino", columns="Variable", values="pct")[
    ["Delta_VPN_BESS_MUSD", "Delta_VPN_BESS_MUSD_ABD"]
]
orden_ef = [
    "Estrategia", "Estrategia x K", "K", "Senal",
    "Sitio", "Senal x Estrategia", "Sitio x K", "Senal x K"
]
piv_ef = piv_ef.reindex([e for e in orden_ef if e in piv_ef.index]).reset_index()
piv_ef.columns = ["Factor / Interacción", "5 estrategias (A, B, C, D, F) [%]", "Subfactorial usuario (A, B, D) [%]"]
print("\nParticipación porcentual en la suma de cuadrados de ΔVPN_BESS (%):")
mostrar_tabla(piv_ef, dec=2)

# Figura 15_1: Panel (a) Delta_VPN_BESS por estrategia y K; Panel (b) Descomposición de varianza
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.4, 4.6), layout="constrained")

res_ek = FAC_NB08.groupby(["K", "Estrategia"])["Delta_VPN_BESS_MUSD"].mean().unstack("Estrategia").loc[["K1", "K3", "K2", "K4"]]
x_k = np.arange(4)
ancho = 0.15
ests_orden = ["A Autoconsumo", "B Merito diario", "D Merito+Export", "C Merchant", "F Merchant reserva"]
for idx, est in enumerate(ests_orden):
    corto = est.split()[0]
    vals = res_ek[est].to_numpy()
    ax1.bar(x_k + (idx - 2) * ancho, vals, width=ancho, color=PALETA[corto], label=f"{corto} ({est.split(' ', 1)[1]})")
ax1.axhline(0, color="#333333", lw=0.9, ls="--")
ax1.set_xticks(x_k)
ax1.set_xticklabels(["K1\n(50/400, r bajo)", "K3 (Ref.)\n(45/280)", "K2\n(45/180)", "K4\n(50/100, r alto)"])
ax1.set_ylim(-56, 45)
ax1.set_ylabel("ΔVPN medio del BESS [MUSD]")
ax1.set_title("a) Valor incremental medio del BESS por configuración y regla", loc="left", fontweight="bold")
ax1.legend(frameon=False, fontsize=8, ncol=2, loc="lower left")

y_pos = np.arange(len(piv_ef))
h_bar = 0.36
ax2.barh(y_pos - h_bar/2, piv_ef["5 estrategias (A, B, C, D, F) [%]"], height=h_bar, color="#7048c0", label="Completo (A, B, C, D, F)")
ax2.barh(y_pos + h_bar/2, piv_ef["Subfactorial usuario (A, B, D) [%]"], height=h_bar, color="#1baf7a", label="Orientadas al usuario (A, B, D)")
for i, (v1, v2) in enumerate(zip(piv_ef["5 estrategias (A, B, C, D, F) [%]"], piv_ef["Subfactorial usuario (A, B, D) [%]"])):
    if v1 >= 1.0:
        ax2.text(v1 + 1.0, i - h_bar/2, f"{es(v1, 1)} %", va="center", fontsize=8, color="#7048c0")
    if v2 >= 1.0:
        ax2.text(v2 + 1.0, i + h_bar/2, f"{es(v2, 1)} %", va="center", fontsize=8, color="#0d5f4f")
ax2.set_yticks(y_pos)
ax2.set_yticklabels(piv_ef["Factor / Interacción"])
ax2.invert_yaxis()
ax2.set_xlim(0, 108)
ax2.set_xlabel("Fracción de suma de cuadrados η² [%]")
ax2.set_title("b) Qué explica la variación de ΔVPN_BESS", loc="left", fontweight="bold")
ax2.legend(frameon=False, loc="lower right", fontsize=8.5)

guardar_fig(fig, "15_1_etapa1_factorial_sin_termica")

Resumen de las 4 configuraciones categóricas K1–K4:


| K | f_PV | P_BESS_MW | E_BESS_MWh | Duracion_h | r_medio | Delta_VPN_medio_ABD_MUSD | Régimen |
|---|---:|---:|---:|---:|---:|---:|---|
| K1 | 1,000 | 50,000 | 400,000 | 8,000 | 0,774 | 27,155 | Almacenamiento abundante |
| K2 | 0,918 | 45,000 | 180,000 | 4,000 | 1,500 | 26,733 | Intermedio |
| K3 | 1,000 | 45,000 | 280,000 | 6,222 | 1,106 | 37,114 | Configuración representativa |
| K4 | 0,830 | 50,000 | 100,000 | 2,000 | 2,280 | 19,630 | Almacenamiento escaso |


Participación porcentual en la suma de cuadrados de ΔVPN_BESS (%):


| Factor / Interacción | 5 estrategias (A, B, C, D, F) [%] | Subfactorial usuario (A, B, D) [%] |
|---|---:|---:|
| Estrategia | 65,73 | 0,07 |
| Estrategia x K | 21,95 | 0,09 |
| K | 8,32 | 66,38 |
| Senal | 1,68 | 28,02 |
| Sitio | 0,16 | 4,11 |
| Senal x Estrategia | 1,77 | 0,02 |
| Sitio x K | 0,03 | 0,92 |
| Senal x K | 0,04 | 0,27 |

## 4 · Contraste contra previsión perfecta sin térmica (`OPT_B` y `OPT_FULL`)

Para separar cuánto pierde una regla heurística por **no conocer el futuro** frente a cuánto se gana por **ampliar las acciones permitidas**, el cuaderno 09 evalúa dos benchmarks de programación lineal/entera mixta sobre seis contextos estratégicos:

1. **`OPT_B` (mismo espacio de acciones que `B`, con previsión perfecta de las 8.760 horas):** carga únicamente desde excedente solar y descarga únicamente hacia la demanda del usuario.
2. **`OPT_FULL` (espacio operativo ampliado, con previsión perfecta):** permite además cargar el BESS desde la red y exportar energía desde el BESS hacia la red.

Esto define tres brechas con identidad algebraica exacta cuando `B` es la mejor heurística:

$$\underbrace{\text{Gap decisión}}_{\text{VPN}_{\text{OPT\_B}} - \text{VPN}_B} \;+\; \underbrace{\text{Gap flexibilidad}}_{\text{VPN}_{\text{OPT\_FULL}} - \text{VPN}_{\text{OPT\_B}}} \;=\; \text{VPN}_{\text{OPT\_FULL}} - \text{VPN}_B$$

In [4]:
BM_NB09 = pd.read_csv(ROOT / "resultados_nb08/benchmarks_operativos_TRM_actual.csv")
GA_NB09 = pd.read_csv(ROOT / "resultados_nb08/gaps_benchmark_TRM_actual.csv")

assert (BM_NB09[BM_NB09.Variante.isin(["OPT_B", "OPT_FULL"])]["estado_solver"] == "optimal").all()
assert GA_NB09["identidad_residuo_COP"].max() < 1.0

tabla_bm09 = GA_NB09[[
    "Contexto", "Sitio", "Senal", "K", "Mejor_heuristica",
    "VPN_mejor_heuristica_MUSD", "VPN_OPT_B_MUSD", "VPN_OPT_FULL_MUSD",
    "Gap_decision_B_pct", "Gap_flex_pct", "Gap_total_pct"
]].copy()
tabla_bm09.columns = [
    "Ctx", "Sitio", "Señal", "K", "Mejor Heur.",
    "VPN Mejor Heur. [MUSD]", "VPN OPT_B [MUSD]", "VPN OPT_FULL [MUSD]",
    "Gap Decisión [%]", "Gap Flexibilidad [%]", "Gap Total [%]"
]
print("Contraste de las reglas heurísticas frente a los benchmarks de previsión perfecta (6 contextos):")
mostrar_tabla(tabla_bm09, dec=2)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.4, 4.4), layout="constrained")
x_c = np.arange(len(GA_NB09))
w_c = 0.26
etiquetas_ctx = [f"{r.Contexto}\n{r.Sitio[:4]}.·{r.Senal}·{r.K}" for r in GA_NB09.itertuples()]

ax1.bar(x_c - w_c, GA_NB09.VPN_mejor_heuristica_MUSD, w_c, color="#eb6834", label="Mejor heurística")
ax1.bar(x_c, GA_NB09.VPN_OPT_B_MUSD, w_c, color="#8b8a83", label="OPT_B (mismo espacio)")
ax1.bar(x_c + w_c, GA_NB09.VPN_OPT_FULL_MUSD, w_c, color="#52514e", label="OPT_FULL (espacio ampliado)")
for i, r in enumerate(GA_NB09.itertuples()):
    ax1.text(x_c[i] - w_c, r.VPN_mejor_heuristica_MUSD + 1.2, r.Mejor_heuristica,
             ha="center", fontsize=8, fontweight="bold", color="#eb6834")
ax1.set_xticks(x_c)
ax1.set_xticklabels(etiquetas_ctx, fontsize=8)
ax1.set_ylim(0, GA_NB09.VPN_OPT_FULL_MUSD.max() * 1.16)
ax1.set_ylabel("VPN a 20 años [MUSD]")
ax1.set_title("a) VPN alcanzado: mejor heurística vs previsión perfecta", loc="left", fontweight="bold")
ax1.legend(frameon=False, fontsize=8, loc="upper left")

ax2.bar(x_c - w_c, GA_NB09.Gap_decision_B_pct, w_c, color="#2a78d6", label="Gap decisión (OPT_B − B)")
ax2.bar(x_c, GA_NB09.Gap_flex_pct, w_c, color="#1baf7a", label="Gap flexibilidad (OPT_FULL − OPT_B)")
ax2.bar(x_c + w_c, GA_NB09.Gap_total_pct, w_c, color="#52514e", label="Gap total (OPT_FULL − Mejor heur.)")
for i, r in enumerate(GA_NB09.itertuples()):
    ax2.text(x_c[i] - w_c, r.Gap_decision_B_pct + 0.08, es(r.Gap_decision_B_pct, 2), ha="center", fontsize=7.5)
    ax2.text(x_c[i], r.Gap_flex_pct + 0.08, es(r.Gap_flex_pct, 2), ha="center", fontsize=7.5)
    ax2.text(x_c[i] + w_c, r.Gap_total_pct + 0.08, es(r.Gap_total_pct, 2), ha="center", fontsize=7.5)
ax2.set_xticks(x_c)
ax2.set_xticklabels(etiquetas_ctx, fontsize=8)
ax2.set_ylim(0, GA_NB09.Gap_total_pct.max() * 1.22)
ax2.set_ylabel("Brecha [% del VPN de referencia]")
ax2.set_title("b) Descomposición de la brecha: decisión vs flexibilidad", loc="left", fontweight="bold")
ax2.legend(frameon=False, fontsize=8, loc="upper left")

guardar_fig(fig, "15_2_etapa1_benchmarks_gaps")

Contraste de las reglas heurísticas frente a los benchmarks de previsión perfecta (6 contextos):


| Ctx | Sitio | Señal | K | Mejor Heur. | VPN Mejor Heur. [MUSD] | VPN OPT_B [MUSD] | VPN OPT_FULL [MUSD] | Gap Decisión [%] | Gap Flexibilidad [%] | Gap Total [%] |
|---|---|---|---|---|---:|---:|---:|---:|---:|---:|
| R1 | Oriente | T0 | K3 | B | 91,44 | 92,05 | 94,11 | 0,67 | 2,24 | 2,84 |
| R2 | Oriente | T1 | K3 | B | 96,04 | 97,21 | 99,91 | 1,20 | 2,79 | 3,88 |
| R3 | Oriente | T0 | K4 | A | 76,75 | 77,01 | 78,59 | 0,35 | 2,05 | 2,34 |
| R4 | Caribe | T1 | K3 | B | 110,78 | 111,83 | 114,51 | 0,93 | 2,40 | 3,25 |
| R5 | Suroccidente | T0 | K1 | B | 93,19 | 94,25 | 97,37 | 1,13 | 3,31 | 4,29 |
| R6 | Antioquia | T0 | K1 | A | 80,04 | 81,15 | 84,22 | 1,43 | 3,79 | 4,97 |

---

# Etapa II · Integración con Autogeneración Térmica (`PV + BESS + G1–G4 + Red`)

## 5 · Formulación del compromiso térmico diario y simulación anual en vivo en K3 (`8.760 h`)

En la segunda etapa se incorpora la planta de autogeneración térmica existente del usuario industrial: **cuatro unidades a gas natural (`G1–G4`)** con mínimos técnicos ($2,40;\; 2,40;\; 1,80;\; 0,96\text{ MW}$) y potencias disponibles ($8,0;\; 8,0;\; 6,0;\; 3,2\text{ MW}$; $25,2\text{ MW}$ en total), consumos específicos diferenciados ($9,638;\; 9,638;\; 11,089;\; 12,240\text{ MMBtu/MWh}$), CVOM por unidad ($9,28\text{ USD/MWh}$ para G1/G2 y $15,32\text{ USD/MWh}$ para G3/G4), gas a $20\text{ USD/MMBtu}$, cargo supuesto de $100\text{ USD/arranque}$ y restricción de **máximo un arranque y una parada por unidad y día calendario**.

### 5.1 · El gestor térmico diario (`evaluador_dia`) y las políticas integradas

Para cualquier perfil horario de demanda residual $r_t \ge 0$ dentro de un día, el gestor térmico resuelve mediante programación dinámica exacta sobre los $6^4 = 1.296$ estados alcanzables:

$$\min_{u_{j,t},\, p_{j,t},\, g_t} \sum_{t} \left[ \sum_{j=1}^{4} \text{CV}_j \cdot p_{j,t} + \lambda_t^{\text{imp}} \cdot g_t + \sum_{j=1}^{4} K_{\text{arr}} \cdot \alpha_{j,t} \right] \quad \text{s.a.} \quad \sum_{j=1}^{4} p_{j,t} + g_t = r_t, \quad P_{j,\min} u_{j,t} \le p_{j,t} \le P_{j,\max} u_{j,t}$$

Sobre este gestor común operan las cinco políticas integradas:
- **`A_INT` (Autoconsumo integrado):** el BESS despacha con la regla causal de autoconsumo `A` (priorizando cubrir déficit con energía solar almacenada) y el gestor térmico despacha día a día el residual resultante.
- **`B_INT` (Mérito diario integrado):** construye sobre la referencia térmica del día candidatos divisibles por hora (`S`, que reducen generación por encima del mínimo técnico o importación de red) y candidatos indivisibles de bloque (`U`, que apagan por completo una unidad que arranca en el día ahorrando además su cargo de arranque), seleccionándolos por densidad de ahorro en USD/MWh de SOC.
- **`D_INT` (Mérito + Exportación integrada):** extiende `B_INT` comparando en cada hora con excedente el ingreso de exportar desde el BESS frente al valor marginal de conservar esa energía para sustituir suministro térmico/red más tarde en el día o en la madrugada siguiente.
- **`C_INT` y `F_INT` (Bloque adicional con carga desde red):** permiten comprar energía de la red en las horas del primer cuartil tarifario del día para completar paquetes de descarga hacia demanda o exportación (`F_INT` limita esas compras al cupo diario no cubierto por el excedente solar).

### 5.2 · Simulación anual en vivo del bloque principal (`8.760 horas` en K3, `T0` y `T1`)

La celda siguiente **ejecuta en vivo las 10 trayectorias anuales completas de 8.760 horas** (`RED_TERMICA`, `PV_SIN_BESS`, `A_INT`, `B_INT`, `D_INT` bajo `T0` y `T1`) en el escenario vigente de la versión 1 (`C1_v1_p0`: gas 20 USD/MMBtu, CVOM por unidad, sin carbono ni cortes de disponibilidad), valida todas las restricciones físicas e identidades económicas con `termica_vpn.auditar`, y comprueba que reproducen exactamente los resultados oficiales de `resultados_correccion_numerica/20260920/`. Además, ejecuta en vivo una ventana de 72 h de `C_INT` y `F_INT`.

In [5]:
import termica_carbono_anual as TCA
import termica_cf as CF
import termica_politicas as POL

t0_etapa2 = time.perf_counter()
ctx_tca = TCA.contexto()

# Leemos la tabla oficial del cuaderno 14 (cierre de precisión numérica 20/09/2026)
COM_NB14 = pd.read_csv(
    ROOT / "resultados_correccion_numerica/20260920/notebook14/14_comparacion_conjunta.csv"
).set_index(["Senal", "Caso"])
RES_PILOTO_V2 = pd.read_csv(
    ROOT / "resultados_carbono_v2/piloto_K3_20260926_tablas_v2/resumen_anual.csv"
)
REF_C1 = RES_PILOTO_V2[RES_PILOTO_V2.Escenario_piloto == "C1_v1_p0"].set_index(["Senal", "Caso"])

casos_v1_vivo = [("T0", c) for c in ("RED_TERMICA", "PV_SIN_BESS", "A_INT", "B_INT", "D_INT")] + \
                [("T1", c) for c in ("RED_TERMICA", "PV_SIN_BESS", "A_INT", "B_INT", "D_INT")]

filas_v1_vivo = []
TRAYECTORIAS_V1_VIVO = {}

for senal, caso in casos_v1_vivo:
    t_c = time.perf_counter()
    out = TCA.resolver("C1_v1_p0", senal, caso, 600, 1e-6)
    dt_c = time.perf_counter() - t_c
    assert out["estado"] == "OK", f"Fallo en {senal}/{caso}: {out['estado']}"
    r = out["resultado"]
    k = out["contabilidad"]
    TRAYECTORIAS_V1_VIVO[senal, caso] = r

    # Verificación estricta contra las referencias congeladas del repositorio
    ref = REF_C1.loc[(senal, caso)]
    err_usd = abs(k["Costo_financiero_USD"] - float(ref.Costo_financiero_USD))
    assert err_usd < 1e-5, f"Diferencia de costo en {senal}/{caso}: {err_usd} USD"
    if (senal, caso) in COM_NB14.index:
        err_nb14 = abs(k["Costo_financiero_USD"] - float(COM_NB14.loc[(senal, caso), "Costo_USD"]))
        assert err_nb14 < 1e-5, f"Diferencia frente a cuaderno 14 en {senal}/{caso}: {err_nb14} USD"

    cierre_soc = float(r["SOC"][-1] - r["SOC_inicial"]) if "SOC" in r and len(r["SOC"]) else 0.0
    assert abs(cierre_soc) <= 1e-4, f"Cierre SOC fuera de tolerancia: {cierre_soc}"

    filas_v1_vivo.append({
        "Señal": senal,
        "Caso": caso,
        "Costo neto [MUSD/año]": k["Costo_financiero_USD"] / 1e6,
        "Gas + CVOM [MUSD]": (k["Combustible_USD"] + k["CVOM_USD"]) / 1e6,
        "Compras red [MUSD]": k["Compras_red_financiero_USD"] / 1e6,
        "Ingreso exp. [MUSD]": k["Ingreso_exportacion_USD"] / 1e6,
        "Térmica [GWh]": k["Generacion_termica_MWh"] / 1e3,
        "Red [GWh]": k["Importacion_MWh"] / 1e3,
        "Arranques [1/año]": int(k["Arranques"]),
        "CO₂ operativo [kt]": k["Emisiones_operativas_t"] / 1e3,
        "Tiempo [s]": dt_c,
        "Error vs Ref. [USD]": err_usd,
    })

# Demostración en vivo de C_INT y F_INT con cierre cíclico sobre ventana de 72 h (enero 2025)
f_72 = ctx_tca["fechas"][:72]
pv_k3, b_k3, pe_k3, _ = TCA.dimensionar()
pv_72 = pv_k3[:72]
load_72 = np.asarray(ctx_tca["E"]["LOAD"][:72], float)
ci_72 = np.asarray(ctx_tca["C_IMP_COP"]["T0"][:72], float) / ctx_tca["TRM"]
ce_72 = np.asarray(ctx_tca["C_EXP_COP"][:72], float) / ctx_tca["TRM"]
res_c72 = CF.ciclo_cf(f_72, pv_72, load_72, ci_72, ce_72, b_k3, pe_k3, ctx_tca["P"], 100.0, "libre", reserva=False)["resultado"]
res_f72 = CF.ciclo_cf(f_72, pv_72, load_72, ci_72, ce_72, b_k3, pe_k3, ctx_tca["P"], 100.0, "libre", reserva=True)["resultado"]
val_c72 = CF.validar_cf(res_c72, ctx_tca["P"])
val_f72 = CF.validar_cf(res_f72, ctx_tca["P"])
assert val_c72["Estado"] == "OK" and val_f72["Estado"] == "OK"

dt_etapa2 = time.perf_counter() - t0_etapa2
DF_V1_VIVO = pd.DataFrame(filas_v1_vivo)
print(f"Simulación anual térmica en vivo completada en {es(dt_etapa2, 2)} s "
      f"(10 trayectorias × 8.760 h + demostración C_INT/F_INT 72 h validadas).")
print(f"Error máximo frente al cierre oficial del cuaderno 14: {DF_V1_VIVO['Error vs Ref. [USD]'].max():.2e} USD.")
mostrar_tabla(DF_V1_VIVO.drop(columns=["Error vs Ref. [USD]"]), dec=3)

,mín [COP/kWh],media [COP/kWh],máx [COP/kWh],rango máx/mín,costo demanda [mil MM COP]
T0 — señal cuasi-plana,653.2,657.4,659.9,1.01x,104.36
T1 — señal horaria indexada,510.2,653.2,"2,636.4",5.17x,104.36


Tarifa,Estrategia,Descarga [MWh],Export BESS [MWh],Carga red [MWh],Ciclos/día,Precio medio descarga,Validación
T0 — señal cuasi-plana,A Autoconsumo,"31,689",0,0,0.90,659.0,OK
T0 — señal cuasi-plana,B Merito diario,"31,672",0,0,0.90,659.2,OK
T0 — señal cuasi-plana,C Merchant,"30,484",300,"5,834",0.87,659.9,OK
T0 — señal cuasi-plana,D Merito+Export,"31,747",180,0,0.91,659.2,OK
T1 — señal horaria indexada,A Autoconsumo,"31,689",0,0,0.90,761.1,OK
T1 — señal horaria indexada,B Merito diario,"31,497",0,0,0.90,769.6,OK
T1 — señal horaria indexada,C Merchant,"32,004",0,"21,083",0.91,770.4,OK
T1 — señal horaria indexada,D Merito+Export,"31,497",0,0,0.90,769.6,OK


,mín [COP/kWh],media [COP/kWh],máx [COP/kWh],rango máx/mín,costo demanda [mil MM COP]
T0 — señal cuasi-plana,653.2,657.4,659.9,1.01x,104.36
T1 — señal horaria indexada,510.2,653.2,"2,636.4",5.17x,104.36


Tarifa,Estrategia,Descarga [MWh],Export BESS [MWh],Carga red [MWh],Ciclos/día,Precio medio descarga,Validación
T0 — señal cuasi-plana,A Autoconsumo,"31,689",0,0,0.90,659.0,OK
T0 — señal cuasi-plana,B Merito diario,"31,672",0,0,0.90,659.2,OK
T0 — señal cuasi-plana,C Merchant,"30,484",300,"5,834",0.87,659.9,OK
T0 — señal cuasi-plana,D Merito+Export,"31,747",180,0,0.91,659.2,OK
T1 — señal horaria indexada,A Autoconsumo,"31,689",0,0,0.90,761.1,OK
T1 — señal horaria indexada,B Merito diario,"31,497",0,0,0.90,769.6,OK
T1 — señal horaria indexada,C Merchant,"32,004",0,"21,083",0.91,770.4,OK
T1 — señal horaria indexada,D Merito+Export,"31,497",0,0,0.90,769.6,OK


Simulación anual térmica en vivo completada en 33,01 s (10 trayectorias × 8.760 h + demostración C_INT/F_INT 72 h validadas).
Error máximo frente al cierre oficial del cuaderno 14: 3.73e-09 USD.


| Señal | Caso | Costo neto [MUSD/año] | Gas + CVOM [MUSD] | Compras red [MUSD] | Ingreso exp. [MUSD] | Térmica [GWh] | Red [GWh] | Arranques [1/año] | CO₂ operativo [kt] | Tiempo [s] |
|---|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| T0 | RED_TERMICA | 32,270 | 27,283 | 4,983 | 0,000 | 135,039 | 23,704 | 37 | 78,564 | 0,508 |
| T0 | PV_SIN_BESS | 14,697 | 16,411 | 3,091 | 4,877 | 81,224 | 14,743 | 723 | 47,302 | 0,149 |
| T0 | A_INT | 4,740 | 4,727 | 1,508 | 1,553 | 23,395 | 7,203 | 578 | 13,911 | 0,368 |
| T0 | B_INT | 5,024 | 6,236 | 0,411 | 1,669 | 30,865 | 1,967 | 458 | 17,622 | 3,197 |
| T0 | D_INT | 5,158 | 6,450 | 0,427 | 1,765 | 31,925 | 2,040 | 463 | 18,228 | 8,433 |
| T1 | RED_TERMICA | 29,661 | 12,015 | 17,595 | 0,000 | 58,526 | 100,217 | 510 | 43,239 | 0,196 |
| T1 | PV_SIN_BESS | 13,172 | 7,513 | 10,464 | 4,877 | 36,562 | 59,405 | 712 | 26,720 | 0,151 |
| T1 | A_INT | 4,044 | 1,393 | 4,179 | 1,553 | 6,805 | 23,793 | 245 | 6,196 | 0,349 |
| T1 | B_INT | 4,166 | 2,294 | 3,524 | 1,682 | 11,338 | 20,514 | 303 | 8,401 | 3,176 |
| T1 | D_INT | 4,225 | 2,384 | 3,553 | 1,742 | 11,782 | 20,668 | 313 | 8,667 | 9,229 |

## 6 · Comparación anual conjunta `A/B/D/C/F` y cotas de los benchmarks (`OPT_AB`, `OPT_D`, `OPT_CF`)

Al integrar el bloque principal (`A_INT`, `B_INT`, `D_INT` y sus benchmarks `OPT_AB` y `OPT_D`) con el bloque de carga desde red autorizada (`C_INT`, `F_INT` y su benchmark `OPT_CF`) sobre las 8.760 horas de 2025 en K3, se obtienen tres conclusiones operativas fundamentales:

1. **Orden de las reglas sin carga desde red (`A_INT < B_INT < D_INT`):** bajo ambas señales (`T0` y `T1`), el autoconsumo inmediato (`A_INT`) logra el menor costo operativo neto ($4,7396\text{ MUSD/año}$ en `T0` y $4,0439\text{ MUSD/año}$ en `T1`), reduciendo frente a `PV_SIN_BESS` el costo operativo anual entre $67,75\text{–}69,30\,\%$.
2. **Desempeño de `C_INT` y `F_INT` con térmica:** bajo `T0` ambas reglas comparten los indicadores anuales y no cargan desde red; su costo anual es $11,3670\text{ MUSD}$ y conservan SOC disponible sin descargar en $1.183\text{ h}$ con déficit. Bajo `T1` difieren: `C_INT` carga $52,79\text{ MWh}$ desde red, cuesta $8,0550\text{ MUSD}$ y registra $2.602\text{ h}$ de déficit con SOC sin descargar; `F_INT` carga $39,70\text{ MWh}$, cuesta $8,0553\text{ MUSD}$ y registra $2.616\text{ h}$.
3. **Precisión y lectura de los benchmarks con límite de tiempo (600 s):** cuando el solver MILP alcanza los 600 s sin cerrar la brecha a $10^{-6}$, el óptimo verdadero $C^*$ queda acotado en el intervalo $[\text{Cota inferior},\; \text{Mejor factible conocido}]$, y la distancia de cada política se reporta rigurosamente como intervalo.

In [6]:
COTAS_NB14 = pd.read_csv(
    ROOT / "resultados_correccion_numerica/20260920/notebook14/14_cotas_conjuntas.csv"
).set_index(["Senal", "Benchmark"])
DIST_NB14 = pd.read_csv(
    ROOT / "resultados_correccion_numerica/20260920/notebook14/14_distancias_conjuntas.csv"
)

# Tabla conjunta de las 16 trayectorias integradas (5 políticas + 3 benchmarks × 2 señales)
conj_tabla = COM_NB14.reset_index()[[
    "Senal", "Caso", "Costo_USD", "Costo_termico_USD", "Costo_red_demanda_USD",
    "Ingreso_exportacion_USD", "Arranques", "Termica_MWh", "Importacion_total_MWh",
    "Descarga_demanda_MWh", "Exportacion_total_MWh", "EFC_util_entrega_AC"
]].copy()
for col_usd in ("Costo_USD", "Costo_termico_USD", "Costo_red_demanda_USD", "Ingreso_exportacion_USD"):
    conj_tabla[col_usd] = conj_tabla[col_usd] / 1e6
for col_mwh in ("Termica_MWh", "Importacion_total_MWh", "Descarga_demanda_MWh", "Exportacion_total_MWh"):
    conj_tabla[col_mwh] = conj_tabla[col_mwh] / 1e3
conj_tabla.columns = [
    "Señal", "Caso", "Costo neto [MUSD]", "Térmico [MUSD]", "Red [MUSD]",
    "Ing. Exp. [MUSD]", "Arranques", "Térmica [GWh]", "Import. [GWh]",
    "BESS→Dem [GWh]", "Export. [GWh]", "EFC [ciclos/año]"
]
print("Comparación anual conjunta en K3 (gas 20 USD/MMBtu, CVOM por unidad):")
mostrar_tabla(conj_tabla, dec=3)

# Tabla de distancias a los benchmarks compatibles con intervalos explícitos
dist_vista = DIST_NB14[["Senal", "Politica", "Benchmark", "Costo_USD"]].copy()
dist_vista["Costo política [MUSD]"] = dist_vista["Costo_USD"] / 1e6
dist_vista["Cota–Mejor factible benchmark [MUSD]"] = [
    intervalo_es(COTAS_NB14.loc[(r.Senal, r.Benchmark), "Cota_USD"] / 1e6,
                 COTAS_NB14.loc[(r.Senal, r.Benchmark), "Mejor_factible_USD"] / 1e6, 4)
    for r in DIST_NB14.itertuples()
]
dist_vista["Distancia al óptimo [mil USD/año]"] = [
    intervalo_es(r.Distancia_min_USD / 1e3, r.Distancia_max_USD / 1e3, 2)
    for r in DIST_NB14.itertuples()
]
dist_vista["Distancia relativa [%]"] = [
    intervalo_es(r.Distancia_min_pct, r.Distancia_max_pct, 2)
    for r in DIST_NB14.itertuples()
]
print("\nDistancias de cada política integrada a su benchmark compatible (intervalos garantizados por cota):")
mostrar_tabla(dist_vista.drop(columns=["Costo_USD"]), dec=4)

# Figura 15_3: Costos operativos frente a cotas (izquierda) y Arranques térmicos (derecha)
fig, axs = plt.subplots(1, 2, figsize=(12.5, 5.2), layout="constrained")
casos_orden = ["A_INT", "B_INT", "D_INT", "C_INT", "F_INT", "OPT_AB", "OPT_D", "OPT_CF"]
y_idx = np.arange(len(casos_orden))

ax = axs[0]
for s_i, (senal, offset, marker) in enumerate([("T0", -0.16, "o"), ("T1", +0.16, "s")]):
    for j, c in enumerate(casos_orden):
        color_c = PALETA[c]
        if not c.startswith("OPT"):
            val = COM_NB14.loc[(senal, c), "Costo_USD"] / 1e6
            ax.scatter(val, j + offset, s=55, color=color_c, marker=marker, edgecolor="#222222", lw=0.6, zorder=3)
            ax.annotate(f"{senal}: {es(val, 2)}", (val, j + offset), xytext=(6, -3 if s_i else 2),
                        textcoords="offset points", fontsize=7.8)
        else:
            r_c = COTAS_NB14.loc[(senal, c)]
            lo, hi = r_c.Cota_USD / 1e6, r_c.Mejor_factible_USD / 1e6
            ax.plot([lo, hi], [j + offset, j + offset], color="#52514e", lw=2.8, marker="|", ms=8, zorder=3)
            ax.annotate(f"{senal}: {intervalo_es(lo, hi, 2)}", (hi, j + offset), xytext=(6, -3 if s_i else 2),
                        textcoords="offset points", fontsize=7.8)
ax.axhline(4.5, color="#cccccc", lw=0.9, ls="--")
ax.set_yticks(y_idx)
ax.set_yticklabels(casos_orden)
ax.invert_yaxis()
ax.set_xlim(3.2, 13.2)
ax.set_xlabel("Costo operativo neto anual [MUSD/año]")
ax.set_title("a) Costo anual de las políticas e intervalos de los benchmarks", loc="left", fontweight="bold")
ax.legend(handles=[
    Line2D([], [], marker="o", color="#333333", ls="", label="T0 (regulada)"),
    Line2D([], [], marker="s", color="#333333", ls="", label="T1 (indexada)"),
    Line2D([], [], color="#52514e", lw=2.8, marker="|", label="Cota–Mejor factible"),
], frameon=False, loc="lower right", fontsize=8)

ax2 = axs[1]
x_arr = np.arange(len(casos_orden))
w_arr = 0.36
arr_t0 = [COM_NB14.loc[("T0", c), "Arranques"] for c in casos_orden]
arr_t1 = [COM_NB14.loc[("T1", c), "Arranques"] for c in casos_orden]
ax2.bar(x_arr - w_arr/2, arr_t0, width=w_arr, color="#7048c0", label="T0 (regulada)")
ax2.bar(x_arr + w_arr/2, arr_t1, width=w_arr, color="#c07a00", label="T1 (indexada)")
for i, (v0, v1) in enumerate(zip(arr_t0, arr_t1)):
    ax2.text(x_arr[i] - w_arr/2, v0 + 10, f"{int(v0)}", ha="center", fontsize=7.5)
    ax2.text(x_arr[i] + w_arr/2, v1 + 10, f"{int(v1)}", ha="center", fontsize=7.5)
ax2.set_xticks(x_arr)
ax2.set_xticklabels(casos_orden, rotation=30)
ax2.set_ylim(0, max(arr_t0) * 1.16)
ax2.set_ylabel("Arranques térmicos acumulados [arranques/año]")
ax2.set_title("b) Eventos de arranque térmico anual por política y señal", loc="left", fontweight="bold")
ax2.legend(frameon=False, loc="upper right", fontsize=8.5)

guardar_fig(fig, "15_3_etapa2_comparacion_anual_termica")

Comparación anual conjunta en K3 (gas 20 USD/MMBtu, CVOM por unidad):


| Señal | Caso | Costo neto [MUSD] | Térmico [MUSD] | Red [MUSD] | Ing. Exp. [MUSD] | Arranques | Térmica [GWh] | Import. [GWh] | BESS→Dem [GWh] | Export. [GWh] | EFC [ciclos/año] |
|---|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| T0 | A_INT | 4,740 | 4,727 | 1,508 | 1,553 | 578 | 23,395 | 7,203 | 65,369 | 23,335 | 291,825 |
| T0 | B_INT | 5,024 | 6,236 | 0,411 | 1,669 | 458 | 30,865 | 1,967 | 63,135 | 25,817 | 281,854 |
| T0 | D_INT | 5,158 | 6,450 | 0,427 | 1,765 | 463 | 31,925 | 2,040 | 62,002 | 26,568 | 297,185 |
| T0 | C_INT | 11,367 | 15,172 | 0,258 | 4,131 | 678 | 75,095 | 1,232 | 19,640 | 74,090 | 89,888 |
| T0 | F_INT | 11,367 | 15,172 | 0,258 | 4,131 | 678 | 75,095 | 1,232 | 19,640 | 74,090 | 89,888 |
| T0 | OPT_AB | 4,593 | 6,745 | 0,050 | 2,239 | 361 | 33,387 | 0,240 | 62,340 | 26,700 | 278,303 |
| T0 | OPT_D | 4,569 | 6,768 | 0,224 | 2,459 | 360 | 33,497 | 1,072 | 61,398 | 27,536 | 282,573 |
| T0 | OPT_CF | 4,568 | 6,797 | 0,194 | 2,481 | 362 | 33,644 | 1,031 | 61,397 | 27,631 | 282,993 |
| T1 | A_INT | 4,044 | 1,393 | 4,179 | 1,553 | 245 | 6,805 | 23,793 | 65,369 | 23,335 | 291,825 |
| T1 | B_INT | 4,166 | 2,294 | 3,524 | 1,682 | 303 | 11,338 | 20,514 | 64,115 | 24,728 | 286,229 |
| T1 | D_INT | 4,225 | 2,384 | 3,553 | 1,742 | 313 | 11,782 | 20,668 | 63,516 | 25,083 | 296,011 |
| T1 | C_INT | 8,055 | 5,300 | 6,516 | 3,812 | 414 | 26,083 | 37,518 | 32,419 | 59,989 | 145,127 |
| T1 | F_INT | 8,055 | 5,300 | 6,519 | 3,812 | 414 | 26,083 | 37,520 | 32,404 | 59,993 | 145,061 |
| T1 | OPT_AB | 3,853 | 2,015 | 4,057 | 2,230 | 114 | 9,975 | 23,611 | 62,380 | 26,656 | 278,483 |
| T1 | OPT_D | 3,837 | 2,074 | 4,059 | 2,308 | 118 | 10,264 | 23,624 | 62,079 | 26,842 | 283,088 |
| T1 | OPT_CF | 3,817 | 1,881 | 3,977 | 2,309 | 106 | 9,311 | 24,735 | 63,452 | 26,848 | 289,244 |


Distancias de cada política integrada a su benchmark compatible (intervalos garantizados por cota):


| Senal | Politica | Benchmark | Costo política [MUSD] | Cota–Mejor factible benchmark [MUSD] | Distancia al óptimo [mil USD/año] | Distancia relativa [%] |
|---|---|---|---:|---|---|---|
| T0 | A_INT | OPT_AB | 4,7396 | [4,5906; 4,5928] | [146,87; 149,01] | [3,19; 3,25] |
| T0 | B_INT | OPT_AB | 5,0237 | [4,5906; 4,5928] | [430,94; 433,08] | [9,38; 9,44] |
| T0 | D_INT | OPT_D | 5,1581 | [4,5654; 4,5689] | [589,23; 592,62] | [12,89; 12,99] |
| T0 | C_INT | OPT_CF | 11,3670 | [4,5643; 4,5680] | [6.799,10; 6.802,68] | [148,84; 149,04] |
| T0 | F_INT | OPT_CF | 11,3670 | [4,5643; 4,5680] | [6.799,10; 6.802,68] | [148,84; 149,04] |
| T1 | A_INT | OPT_AB | 4,0439 | [3,8525; 3,8531] | [190,84; 191,27] | [4,95; 4,97] |
| T1 | B_INT | OPT_AB | 4,1665 | [3,8525; 3,8531] | [313,48; 313,91] | [8,13; 8,15] |
| T1 | D_INT | OPT_D | 4,2254 | [3,8362; 3,8366] | [388,77; 389,10] | [10,13; 10,15] |
| T1 | C_INT | OPT_CF | 8,0550 | [3,8163; 3,8168] | [4.238,32; 4.238,74] | [111,04; 111,07] |
| T1 | F_INT | OPT_CF | 8,0553 | [3,8163; 3,8168] | [4.238,55; 4.238,98] | [111,05; 111,08] |

### 6.1 · Dinámica horaria en una semana representativa (generación integrada y SOC)

A partir de las trayectorias anuales **simuladas en vivo en la §5.2**, la siguiente figura muestra la operación horaria durante la semana representativa del 21 al 27 de abril de 2025 bajo `T0`, permitiendo observar cómo `A_INT` descarga inmediatamente al caer el sol mientras que `B_INT` y `D_INT` modulan la descarga y el encendido de `G1–G2`.

In [7]:
# Graficamos directamente desde las trayectorias anuales calculadas en vivo (TRAYECTORIAS_V1_VIVO)
fechas_anual = ctx_tca["fechas"]
ini_sem = pd.Timestamp("2025-04-21 00:00:00")
fin_sem = ini_sem + pd.Timedelta(days=7)
mask_sem = (fechas_anual >= ini_sem) & (fechas_anual < fin_sem)
t_sem = fechas_anual[mask_sem]

fig, axes = plt.subplots(4, 1, figsize=(12.0, 8.2), sharex=True, layout="constrained",
                         gridspec_kw={"height_ratios": [1, 1, 1, 0.85]})
colores_capas = ["#eda100", "#2a78d6", "#7048c0", "#8b8a83"]
nombres_capas = ["PV directo a demanda", "Descarga BESS a demanda", "Generación térmica (G1–G4)", "Importación de red"]

for idx, regla in enumerate(("A_INT", "B_INT", "D_INT")):
    ax = axes[idx]
    tr = TRAYECTORIAS_V1_VIVO["T0", regla]
    pv_dir = np.minimum(tr["PV"][mask_sem], tr["Carga"][mask_sem])
    bess_dem = tr["Descarga_demanda"][mask_sem]
    term_tot = tr["potencia"][mask_sem, :4].sum(axis=1)
    red_dem = tr["potencia"][mask_sem, 4]
    demanda_sem = tr["Carga"][mask_sem]

    ax.stackplot(t_sem, pv_dir, bess_dem, term_tot, red_dem, colors=colores_capas, labels=nombres_capas, alpha=0.88)
    ax.plot(t_sem, demanda_sem, color="#111111", lw=1.1, label="Demanda industrial")
    ax.set_ylabel("Potencia [MW]")
    ax.set_ylim(0, 29 if idx == 0 else 25)
    ax.set_title(f"{regla} · Suministro horario de la demanda bajo T0 (semana 21–27 abr. 2025)", loc="left", fontweight="bold", fontsize=9.5)
    if idx == 0:
        ax.legend(frameon=False, ncol=5, loc="upper right", fontsize=8)

ax_soc = axes[3]
for regla, ls in [("A_INT", "-"), ("B_INT", "--"), ("D_INT", ":")]:
    soc_sem = TRAYECTORIAS_V1_VIVO["T0", regla]["SOC"][mask_sem]
    ax_soc.plot(t_sem, soc_sem, color=PALETA[regla], lw=1.8, ls=ls, label=f"SOC {regla}")
ax_soc.axhline(56.0, color="#888888", lw=0.8, ls="--", label="SOC mín. (56 MWh)")
ax_soc.axhline(280.0, color="#888888", lw=0.8, ls=":", label="SOC máx. (280 MWh)")
ax_soc.set_ylabel("SOC [MWh]")
ax_soc.set_ylim(0, 345)
ax_soc.set_title("Estado de carga (SOC) del BESS durante la semana representativa", loc="left", fontweight="bold", fontsize=9.5)
ax_soc.legend(frameon=False, ncol=5, loc="upper right", fontsize=8)
ax_soc.xaxis.set_major_formatter(mdates.DateFormatter("%d/%m\n%Hh"))

guardar_fig(fig, "15_4_etapa2_despacho_semanal_T0")

## 7 · Factorial anual térmico (`K1–K4`) y evaluación financiera multianual de K3 (20 años)

### 7.1 · ¿Se conserva el orden de las reglas al cambiar la configuración `K1–K4`?

El experimento factorial térmico evalúa las cuatro configuraciones congeladas (`K1–K4`) bajo `T0` y `T1` con las cinco políticas (`A_INT`, `B_INT`, `D_INT`, `C_INT`, `F_INT`) y sus benchmarks compatibles (72 casos anuales en total):
- En **`K1` y `K3`** (baterías grandes de $400$ y $280\text{ MWh}$ con $f_{\text{PV}} = 1,0$), **`A_INT` tiene el menor costo operativo anual** bajo ambas tarifas.
- En **`K2` y `K4`** (baterías de menor energía, $180$ y $100\text{ MWh}$, con $f_{\text{PV}} = 0,918$ y $0,830$), **`D_INT` logra el menor costo operativo anual**, superando a `A_INT` entre $0,40\,\%$ y $1,19\,\%$, gracias a una combinación de menor compra de red, menor número de arranques y mayores ingresos por exportación.

### 7.2 · Evaluación financiera multianual de K3 durante 20 años (362 despachos anuales)

Al evaluar las 20 anualidades explícitas de K3 (con degradación solar de $0,5\,\%/\text{año}$, WACC real del $10\,\%$, CAPEX PV de $240,64\text{ mil millones COP}$ y CAPEX BESS de $157,61\text{ mil millones COP}$):
- **`A_INT` alcanza el mayor VPN total ($259,49\text{ mil MM COP}$ en `T0` y $209,49\text{ mil MM COP}$ en `T1`) y el mayor VPN incremental del BESS ($81,48$ y $60,14\text{ mil MM COP}$).**
- **`C_INT` y `F_INT` tienen VPN incremental del BESS negativo ($-89,29\text{ mil MM COP}$ en `T0`; $-43,22$ y $-43,23\text{ mil MM COP}$ en `T1`, respectivamente):** aunque el proyecto conjunto `PV + BESS` muestra VPN total positivo gracias al ahorro del PV, añadir la batería bajo `C_INT` o `F_INT` destruye valor frente a quedarse con `PV_SIN_BESS`.

In [8]:
from ampliacion_tesis import generar as generar_ampliacion_v1

AMP_V1, EXACTOS_V1, FUENTES_AMP_V1 = generar_ampliacion_v1(ROOT)
FAC_TERM = pd.read_csv(ROOT / "resultados_correccion_numerica/20260920/factorial/resumen.csv")
VPN_TERM = pd.read_csv(ROOT / "resultados_correccion_numerica/20260920/vpn/vpn_resumen.csv")
VPN_INT_TERM = pd.read_csv(ROOT / "resultados_correccion_numerica/20260920/vpn/vpn_intervalos.csv")

display(Markdown("**Costo operativo neto anual en el factorial térmico K1–K4 [mil millones COP/año]:**\n\n" + AMP_V1["FACTORIAL_COSTOS"]))
display(Markdown("\n**VPN total (frente a Red + Térmica) y VPN incremental del BESS (frente a PV sin BESS) en K3 a 20 años [mil millones COP reales]:**\n\n" + AMP_V1["VPN_TABLA"]))
display(Markdown("\n**Intervalos del VPN incremental compatible en los benchmarks [mil millones COP reales]:**\n\n" + AMP_V1["VPN_BANDAS"]))

# Figura 15_5: Costo operativo en el factorial K1-K4 (izquierda) y VPN en K3 a 20 años (derecha)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.5, 4.8), layout="constrained")

fac_abd = FAC_TERM[FAC_TERM.Caso.isin(["A_INT", "B_INT", "D_INT"])].copy()
fac_abd["Costo_milMM_COP"] = fac_abd["Costo_operativo_USD"] * costos.TRM_COP_USD / 1e9
etiquetas_ks = [f"{k}/{s}" for s in ("T0", "T1") for k in ("K1", "K3", "K2", "K4")]
x_ks = np.arange(len(etiquetas_ks))
w_ks = 0.25
for idx, pol in enumerate(("A_INT", "B_INT", "D_INT")):
    vals = [
        float(fac_abd[(fac_abd.K == k) & (fac_abd.Senal == s) & (fac_abd.Caso == pol)]["Costo_milMM_COP"].iloc[0])
        for s in ("T0", "T1") for k in ("K1", "K3", "K2", "K4")
    ]
    ax1.bar(x_ks + (idx - 1) * w_ks, vals, width=w_ks, color=PALETA[pol], label=pol)
ax1.axvline(3.5, color="#888888", lw=0.9, ls="--")
ax1.set_xticks(x_ks)
ax1.set_xticklabels(etiquetas_ks, fontsize=8.5)
ax1.set_ylim(0, 43)
ax1.set_ylabel("Costo operativo anual [mil millones COP/año]")
ax1.set_title("a) Factorial térmico K1–K4: A_INT gana en K1/K3 y D_INT en K2/K4", loc="left", fontweight="bold")
ax1.legend(frameon=False, ncol=3, loc="upper left", fontsize=8.5)

pols_vpn = ["A_INT", "B_INT", "D_INT", "C_INT", "F_INT"]
x_v = np.arange(len(pols_vpn))
w_v = 0.35
vpn_inc_t0 = [
    float(VPN_TERM[(VPN_TERM.Senal == "T0") & (VPN_TERM.Caso == p) & (VPN_TERM.Referencia == "PV_SIN_BESS")]["VPN_COP"].iloc[0]) / 1e9
    for p in pols_vpn
]
vpn_inc_t1 = [
    float(VPN_TERM[(VPN_TERM.Senal == "T1") & (VPN_TERM.Caso == p) & (VPN_TERM.Referencia == "PV_SIN_BESS")]["VPN_COP"].iloc[0]) / 1e9
    for p in pols_vpn
]
ax2.bar(x_v - w_v/2, vpn_inc_t0, width=w_v, color="#7048c0", label="T0 (VPN incremental BESS)")
ax2.bar(x_v + w_v/2, vpn_inc_t1, width=w_v, color="#c07a00", label="T1 (VPN incremental BESS)")
ax2.axhline(0, color="#222222", lw=1.0, ls="--")
for i, (v0, v1) in enumerate(zip(vpn_inc_t0, vpn_inc_t1)):
    ax2.text(x_v[i] - w_v/2, v0 + (3.0 if v0 >= 0 else -7.5), es(v0, 1), ha="center", fontsize=8)
    ax2.text(x_v[i] + w_v/2, v1 + (3.0 if v1 >= 0 else -7.5), es(v1, 1), ha="center", fontsize=8)
ax2.set_xticks(x_v)
ax2.set_xticklabels(pols_vpn)
ax2.set_ylim(-112, 105)
ax2.set_ylabel("VPN incremental del BESS [mil millones COP]")
ax2.set_title("b) VPN incremental del BESS en K3 (20 años): A/B/D crean valor; C/F destruyen", loc="left", fontweight="bold")
ax2.legend(frameon=False, loc="upper right", fontsize=8.5)

guardar_fig(fig, "15_5_etapa2_factorial_y_vpn_k3")

**Costo operativo neto anual en el factorial térmico K1–K4 [mil millones COP/año]:**

| K / tarifa | A | B | D | C | F | Menor costo |
|---|---|---|---|---|---|---|
| K1 / T0 | 10,91 | 13,28 | 13,84 | 35,53 | 35,53 | A |
| K2 / T0 | 26,40 | 26,23 | 26,20 | 38,08 | 38,09 | D |
| K3 / T0 | 14,82 | 15,70 | 16,12 | 35,53 | 35,53 | A |
| K4 / T0 | 38,26 | 38,08 | 38,03 | 41,76 | 41,77 | D |
| K1 / T1 | 9,27 | 10,38 | 10,83 | 24,82 | 24,82 | A |
| K2 / T1 | 22,97 | 22,88 | 22,88 | 29,22 | 29,23 | D |
| K3 / T1 | 12,64 | 13,02 | 13,21 | 25,18 | 25,18 | A |
| K4 / T1 | 34,17 | 33,78 | 33,76 | 35,67 | 35,71 | D |


**VPN total (frente a Red + Térmica) y VPN incremental del BESS (frente a PV sin BESS) en K3 a 20 años [mil millones COP reales]:**

| Regla | T0 total | T0 BESS | T1 total | T1 BESS |
|---|---|---|---|---|
| A | 259,49 | 81,48 | 209,49 | 60,14 |
| B | 252,73 | 74,72 | 206,58 | 57,24 |
| D | 249,25 | 71,24 | 205,15 | 55,81 |
| C | 88,72 | -89,29 | 106,13 | -43,22 |
| F | 88,72 | -89,29 | 106,12 | -43,23 |


**Intervalos del VPN incremental compatible en los benchmarks [mil millones COP reales]:**

| Espacio compatible | T0 VPN BESS | T1 VPN BESS |
|---|---|---|
| OPT_AB | [85,47; 85,56] | [65,40; 65,43] |
| OPT_D | [86,11; 86,21] | [65,81; 65,85] |
| OPT_CF | [86,11; 86,24] | [66,37; 66,42] |

---

# Etapa III · Extensión V2: Disponibilidad Térmica, Precio Interno de Carbono y Dimensionamiento Integrado (`M01–M12`)

## 8 · Disponibilidad térmica V2, precio interno de carbono y simulación anual en vivo del piloto K3

La tercera etapa (desarrollada en la rama `tesis-v2-parametros`) extiende el análisis integrado incorporando dos elementos de realismo operativo y ambiental y llevando a cabo el dimensionamiento económico integrado:

1. **Calendario horario de disponibilidad térmica V2 (`disponibilidad_termica.py`):** la flota térmica queda indisponible en las **101 horas** en que la demanda industrial cae por debajo del $25\,\%$ de su mediana anual (todas contenidas en las 115 horas de carga baja conservadas) y durante las **336 horas** del mantenimiento programado de planta (del 1 al 14 de septiembre), totalizando **437 horas indisponibles** ($95,01\,\%$ de disponibilidad anual). En una hora indisponible ninguna unidad puede estar encendida; la parada cuenta como evento ordinario y el rearranque posterior paga el cargo de $100\text{ USD}$.
2. **Precio interno de carbono dentro del despacho y en la contabilidad (`carbono.py`):** un precio de escenario $p_C \in \{0, 10, 20, 26, 50\}\text{ USD/t CO}_2\text{e}$ modifica los costos variables térmicos y la señal de importación de red según la modalidad (`gas_y_red` principal con factor XM 2025 de $0,097018\text{ t CO}_2\text{e/MWh}$, o `solo_gas`):

$$\text{CV}_g^{\text{ajustado}} = \text{HR}_g \cdot P_{\text{gas}} + \text{CVOM}_g + p_C \cdot \text{HR}_g \cdot \text{FE}_{\text{gas}}, \qquad c_{\text{imp}}^{\text{ajustado}} = c_{\text{imp}}^{\text{financiero}} + I_{\text{red}} \cdot p_C \cdot \text{FE}_{\text{red}}$$

La función `carbono.contabilidad` reconstruye por separado el **costo financiero** y el **cargo de carbono**, exigiendo que su suma coincida exactamente con el objetivo del motor (sin doble cobro), y calcula el **VPN ajustado** a 20 años:

$$\text{VPN}_{\text{ajustado}} = \text{VPN}_{\text{financiero}} + \text{TRM}\cdot\sum_{y=1}^{20} \frac{p_C \cdot \left(E_{y}^{\text{cobradas, ref}} - E_{y}^{\text{cobradas, proy}}\right)}{(1 + r)^y}$$

### 8.1 · Simulación anual en vivo de K3 bajo `V2_p0` (0 USD/t) y `V2_p50` (50 USD/t)

La celda siguiente **ejecuta en vivo las 20 trayectorias anuales completas de 8.760 horas** (`RED_TERMICA`, `PV_SIN_BESS`, `A_INT`, `B_INT` y `D_INT` bajo `T0` y `T1` para `V2_p0` y `V2_p50`), comprobando que en ninguna de las 437 horas indisponibles hay unidades encendidas y verificando que todos los costos y emisiones coinciden exactamente con `resultados_carbono_v2/piloto_K3_20260926_tablas_v2/resumen_anual.csv`.

In [9]:
t0_etapa3 = time.perf_counter()

# Verificación del calendario de disponibilidad V2
res_disp = DT.resumen(ctx_tca["disp_v2"], ctx_tca["E"]["LOAD"])
assert (res_disp["horas_parada"], res_disp["horas_mantenimiento"], res_disp["horas_indisponibles"]) == (101, 336, 437)

REF_PILOTO = RES_PILOTO_V2.drop_duplicates(subset=["Escenario_piloto", "Senal", "Caso"]).set_index(["Escenario_piloto", "Senal", "Caso"])
casos_v2_vivo = [
    (esc, senal, caso)
    for esc in ("V2_p0", "V2_p50")
    for senal in ("T0", "T1")
    for caso in ("RED_TERMICA", "PV_SIN_BESS", "A_INT", "B_INT", "D_INT")
]

filas_v2_vivo = []
TRAYECTORIAS_V2_VIVO = {}
for esc, senal, caso in casos_v2_vivo:
    t_c = time.perf_counter()
    out = TCA.resolver(esc, senal, caso, 600, 1e-6)
    dt_c = time.perf_counter() - t_c
    assert out["estado"] == "OK", f"Fallo en {esc}/{senal}/{caso}"
    TRAYECTORIAS_V2_VIVO[esc, senal, caso] = out["resultado"]
    k = out["contabilidad"]
    ref = REF_PILOTO.loc[(esc, senal, caso)]

    err_fin = abs(k["Costo_financiero_USD"] - float(ref.Costo_financiero_USD))
    err_adj = abs(k["Costo_ajustado_USD"] - float(ref.Costo_ajustado_USD))
    err_emi = abs(k["Emisiones_operativas_t"] - float(ref.Emisiones_operativas_t))
    assert max(err_fin, err_adj, err_emi) < 1e-4, f"Discrepancia en {esc}/{senal}/{caso}"

    filas_v2_vivo.append({
        "Escenario": esc,
        "Señal": senal,
        "Caso": caso,
        "Precio CO₂ [USD/t]": k["Precio_CO2_USD_t"],
        "Costo financ. [MUSD]": k["Costo_financiero_USD"] / 1e6,
        "Cargo carbono [MUSD]": k["Cargo_carbono_USD"] / 1e6,
        "Costo ajustado [MUSD]": k["Costo_ajustado_USD"] / 1e6,
        "Térmica [GWh]": k["Generacion_termica_MWh"] / 1e3,
        "Red [GWh]": k["Importacion_MWh"] / 1e3,
        "CO₂ térmico [kt]": k["CO2_termico_t"] / 1e3,
        "CO₂e red [kt]": k["CO2e_red_t"] / 1e3,
        "CO₂ total [kt]": k["Emisiones_operativas_t"] / 1e3,
        "Arranques": int(k["Arranques"]),
        "Tiempo [s]": dt_c,
    })

dt_etapa3 = time.perf_counter() - t0_etapa3
DF_V2_VIVO = pd.DataFrame(filas_v2_vivo)
print(f"Simulación anual V2 en vivo completada en {es(dt_etapa3, 2)} s "
      f"(20 trayectorias × 8.760 h = 175.200 horas con disponibilidad V2 y contabilidad de carbono).")
mostrar_tabla(DF_V2_VIVO, dec=3)

# Tabla de VPN a 20 años de K3 con carbono (0 y 50 USD/t)
VPN_K3_V2 = pd.read_csv(ROOT / "resultados_carbono_v2/vpn_K3_20260926/vpn.csv")
vpn_k3_vista = VPN_K3_V2[VPN_K3_V2.Proyecto.isin(["A_INT", "B_INT", "D_INT"])].copy()
vpn_k3_vista = vpn_k3_vista[[
    "Escenario_piloto", "Senal", "Proyecto", "Referencia",
    "VPN_financiero_MUSD", "VPN_ajustado_MUSD", "TIR", "EFC_total"
]].copy()
vpn_k3_vista["TIR"] = vpn_k3_vista["TIR"] * 100.0
vpn_k3_vista.columns = [
    "Escenario", "Señal", "Regla", "Referencia",
    "VPN Financiero [MUSD]", "VPN Ajustado [MUSD]", "TIR [%]", "EFC 20 años"
]
print("\nEvaluación multianual de K3 a 20 años con disponibilidad V2 y carbono (400 anualidades explícitas):")
mostrar_tabla(vpn_k3_vista, dec=2)

# Figura 15_6: Efecto del carbono en K3 sobre el costo ajustado y las emisiones operativas
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.5, 4.6), layout="constrained")
escs_plot = ["V2_p0", "V2_p26", "V2_p50"]
etiquetas_esc = ["0 USD/t\n(V2_p0)", "26 USD/t\n(V2_p26)", "50 USD/t\n(V2_p50)"]
x_e = np.arange(len(escs_plot))

for senal, ls, marker in [("T0", "-", "o"), ("T1", "--", "s")]:
    for regla in ("A_INT", "B_INT", "D_INT"):
        costos_adj = [float(REF_PILOTO.loc[(e, senal, regla), "Costo_ajustado_USD"]) / 1e6 for e in escs_plot]
        emisiones_kt = [float(REF_PILOTO.loc[(e, senal, regla), "Emisiones_operativas_t"]) / 1e3 for e in escs_plot]
        ax1.plot(x_e, costos_adj, color=PALETA[regla], ls=ls, marker=marker, lw=1.8, label=f"{regla} ({senal})")
        ax2.plot(x_e, emisiones_kt, color=PALETA[regla], ls=ls, marker=marker, lw=1.8, label=f"{regla} ({senal})")

ax1.set_xticks(x_e)
ax1.set_xticklabels(etiquetas_esc)
ax1.set_ylim(3.95, 5.48)
ax1.set_ylabel("Costo operativo ajustado [MUSD/año]")
ax1.set_title("a) Costo anual ajustado por carbono en K3 (año 1)", loc="left", fontweight="bold")
ax1.legend(frameon=False, ncol=2, fontsize=8, loc="upper left")

ax2.set_xticks(x_e)
ax2.set_xticklabels(etiquetas_esc)
ax2.set_ylabel("Emisiones operativas (CO₂ térmico + CO₂e red) [kt/año]")
ax2.set_title("b) Desplazamiento térmico y caída de emisiones con carbono", loc="left", fontweight="bold")
ax2.legend(frameon=False, ncol=2, fontsize=8, loc="upper right")

guardar_fig(fig, "15_6_etapa3_piloto_carbono_k3")

Simulación anual V2 en vivo completada en 54,78 s (20 trayectorias × 8.760 h = 175.200 horas con disponibilidad V2 y contabilidad de carbono).


| Escenario | Señal | Caso | Precio CO₂ [USD/t] | Costo financ. [MUSD] | Cargo carbono [MUSD] | Costo ajustado [MUSD] | Térmica [GWh] | Red [GWh] | CO₂ térmico [kt] | CO₂e red [kt] | CO₂ total [kt] | Arranques | Tiempo [s] |
|---|---|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| V2_p0 | T0 | RED_TERMICA | 0,000 | 32,314 | 0,000 | 32,314 | 129,811 | 28,932 | 73,311 | 2,807 | 76,118 | 44 | 0,198 |
| V2_p0 | T0 | PV_SIN_BESS | 0,000 | 14,719 | 0,000 | 14,719 | 78,005 | 17,962 | 44,054 | 1,743 | 45,796 | 694 | 0,149 |
| V2_p0 | T0 | A_INT | 0,000 | 4,742 | 0,000 | 4,742 | 22,880 | 7,718 | 12,922 | 0,749 | 13,670 | 560 | 0,355 |
| V2_p0 | T0 | B_INT | 0,000 | 4,986 | 0,000 | 4,986 | 29,917 | 2,645 | 16,896 | 0,257 | 17,152 | 443 | 3,180 |
| V2_p0 | T0 | D_INT | 0,000 | 5,118 | 0,000 | 5,118 | 30,934 | 2,719 | 17,470 | 0,264 | 17,734 | 448 | 8,557 |
| V2_p0 | T1 | RED_TERMICA | 0,000 | 29,809 | 0,000 | 29,809 | 56,835 | 101,908 | 32,545 | 9,887 | 42,432 | 472 | 0,191 |
| V2_p0 | T1 | PV_SIN_BESS | 0,000 | 13,302 | 0,000 | 13,302 | 35,217 | 60,750 | 20,179 | 5,894 | 26,073 | 672 | 0,146 |
| V2_p0 | T1 | A_INT | 0,000 | 4,044 | 0,000 | 4,044 | 6,778 | 23,820 | 3,872 | 2,311 | 6,183 | 243 | 0,349 |
| V2_p0 | T1 | B_INT | 0,000 | 4,147 | 0,000 | 4,147 | 11,020 | 20,766 | 6,231 | 2,015 | 8,246 | 289 | 3,025 |
| V2_p0 | T1 | D_INT | 0,000 | 4,207 | 0,000 | 4,207 | 11,465 | 20,912 | 6,482 | 2,029 | 8,511 | 299 | 9,355 |
| V2_p50 | T0 | RED_TERMICA | 50,000 | 33,382 | 0,770 | 34,152 | 0,000 | 158,743 | 0,000 | 15,401 | 15,401 | 0 | 0,192 |
| V2_p50 | T0 | PV_SIN_BESS | 50,000 | 15,253 | 0,466 | 15,719 | 0,000 | 95,967 | 0,000 | 9,311 | 9,311 | 0 | 0,150 |
| V2_p50 | T0 | A_INT | 50,000 | 4,856 | 0,148 | 5,005 | 0,000 | 30,598 | 0,000 | 2,969 | 2,969 | 0 | 0,368 |
| V2_p50 | T0 | B_INT | 50,000 | 4,846 | 0,148 | 4,995 | 0,000 | 30,598 | 0,000 | 2,969 | 2,969 | 0 | 3,413 |
| V2_p50 | T0 | D_INT | 50,000 | 4,851 | 0,150 | 5,001 | 0,000 | 30,977 | 0,000 | 3,005 | 3,005 | 0 | 10,863 |
| V2_p50 | T1 | RED_TERMICA | 50,000 | 30,053 | 1,612 | 31,666 | 35,344 | 123,398 | 20,275 | 11,972 | 32,247 | 468 | 0,212 |
| V2_p50 | T1 | PV_SIN_BESS | 50,000 | 13,437 | 1,022 | 14,459 | 23,390 | 72,577 | 13,405 | 7,041 | 20,446 | 576 | 0,154 |
| V2_p50 | T1 | A_INT | 50,000 | 4,069 | 0,242 | 4,311 | 3,909 | 26,689 | 2,243 | 2,589 | 4,832 | 166 | 0,375 |
| V2_p50 | T1 | B_INT | 50,000 | 4,095 | 0,307 | 4,402 | 6,614 | 24,684 | 3,739 | 2,395 | 6,134 | 241 | 3,347 |
| V2_p50 | T1 | D_INT | 50,000 | 4,126 | 0,313 | 4,439 | 6,823 | 24,794 | 3,858 | 2,406 | 6,263 | 246 | 10,192 |


Evaluación multianual de K3 a 20 años con disponibilidad V2 y carbono (400 anualidades explícitas):


| Escenario | Señal | Regla | Referencia | VPN Financiero [MUSD] | VPN Ajustado [MUSD] | TIR [%] | EFC 20 años |
|---|---|---|---|---:|---:|---:|---:|
| V2_p0 | T0 | A_INT | RED_TERMICA | 83,36 | 83,36 | 18,94 | 5.668,19 |
| V2_p0 | T0 | A_INT | PV_SIN_BESS | 26,23 | 26,23 | 17,22 | 5.668,19 |
| V2_p0 | T0 | B_INT | RED_TERMICA | 81,53 | 81,53 | 18,75 | 5.512,92 |
| V2_p0 | T0 | B_INT | PV_SIN_BESS | 24,40 | 24,40 | 16,72 | 5.512,92 |
| V2_p0 | T0 | D_INT | RED_TERMICA | 80,45 | 80,45 | 18,64 | 5.757,80 |
| V2_p0 | T0 | D_INT | PV_SIN_BESS | 23,32 | 23,32 | 16,44 | 5.757,80 |
| V2_p0 | T1 | A_INT | RED_TERMICA | 68,27 | 68,27 | 17,40 | 5.668,19 |
| V2_p0 | T1 | A_INT | PV_SIN_BESS | 20,35 | 20,35 | 15,66 | 5.668,19 |
| V2_p0 | T1 | B_INT | RED_TERMICA | 67,51 | 67,51 | 17,32 | 5.570,96 |
| V2_p0 | T1 | B_INT | PV_SIN_BESS | 19,59 | 19,59 | 15,45 | 5.570,96 |
| V2_p0 | T1 | D_INT | RED_TERMICA | 67,04 | 67,04 | 17,27 | 5.746,53 |
| V2_p0 | T1 | D_INT | PV_SIN_BESS | 19,12 | 19,12 | 15,33 | 5.746,53 |
| V2_p50 | T0 | A_INT | RED_TERMICA | 91,43 | 96,65 | 19,75 | 5.668,19 |
| V2_p50 | T0 | A_INT | PV_SIN_BESS | 29,77 | 32,42 | 18,13 | 5.668,19 |
| V2_p50 | T0 | B_INT | RED_TERMICA | 91,51 | 96,73 | 19,76 | 5.668,19 |
| V2_p50 | T0 | B_INT | PV_SIN_BESS | 29,85 | 32,50 | 18,15 | 5.668,19 |
| V2_p50 | T0 | D_INT | RED_TERMICA | 91,48 | 96,69 | 19,76 | 5.715,04 |
| V2_p50 | T0 | D_INT | PV_SIN_BESS | 29,82 | 32,45 | 18,15 | 5.715,04 |
| V2_p50 | T1 | A_INT | RED_TERMICA | 70,11 | 81,68 | 17,59 | 5.668,19 |
| V2_p50 | T1 | A_INT | PV_SIN_BESS | 21,26 | 27,83 | 15,90 | 5.668,19 |
| V2_p50 | T1 | B_INT | RED_TERMICA | 69,99 | 81,01 | 17,58 | 5.611,63 |
| V2_p50 | T1 | B_INT | PV_SIN_BESS | 21,14 | 27,16 | 15,87 | 5.611,63 |
| V2_p50 | T1 | D_INT | RED_TERMICA | 69,73 | 80,69 | 17,55 | 5.746,68 |
| V2_p50 | T1 | D_INT | PV_SIN_BESS | 20,88 | 26,84 | 15,80 | 5.746,68 |

## 9 · Diagnóstico de apagado anticipado de unidades térmicas heredadas en `B_INT`

### 9.1 · ¿Por qué `B_INT` mantiene encendida a `G2` en su mínimo técnico algunas madrugadas?

En `B_INT`, los candidatos horarios divisibles (`S`) solo sustituyen generación térmica **por encima de los mínimos técnicos**, mientras que los candidatos indivisibles de bloque (`U`) solo evalúan apagar unidades **que arrancan dentro del mismo día** (acreditando su cargo de arranque). Cuando una unidad (como `G2`, con mínimo de $2,4\text{ MW}$) llega encendida desde las 23:00 del día anterior, bajo `T0` sin carbono su costo variable ($202,04\text{ USD/MWh}$) es menor que el de la red ($208,96\text{–}211,11\text{ USD/MWh}$), por lo que la programación dinámica la conserva encendida en su mínimo durante la madrugada mientras el BESS cubre el resto de la demanda.

El diagnóstico clasifica los pares (día, unidad) en cuatro niveles acumulativos:
- **`N1`:** la unidad viene encendida desde las 23:00 y sigue encendida a las 00:00.
- **`N2`:** además opera en o cerca de su mínimo técnico ($\le 1,05 \cdot P_{\min}$) a las 00:00.
- **`N3`:** además el BESS tiene potencia libre y SOC disponible por encima del mínimo en esa hora.
- **`N4`:** además una descarga adicional del BESS puede cubrir toda la generación de esa unidad hasta su parada original sin reasignar descargas ya planeadas (y en la mayoría de los casos el BESS vuelve a saturarse ese mismo día con excedente solar).

La celda siguiente muestra la frecuencia anual de situaciones `N1–N4` y **ejecuta en vivo el contraste de las alternativas `ALT1` y `ALT2`** sobre las ventanas representativas.

In [10]:
import diagnostico_apagado_b as DAB

FREQ_DAB = pd.read_csv(ROOT / "resultados_carbono_v2/diagnostico_apagado_b_20260926/frecuencia.csv")
VENT_DAB = pd.read_csv(ROOT / "resultados_carbono_v2/diagnostico_apagado_b_20260926/ventanas.csv")

# Ejecución en vivo del contraste de ALT1 y ALT2 sobre la trayectoria anual V2_p0/T0/B_INT recién simulada
esc_dab, con_disp_dab = TCA.escenario("V2_p0")
P_dab = C.parametros_ajustados(ctx_tca["P"], esc_dab)
pv_dab, b_dab, pe_dab, _ = TCA.dimensionar(None)
load_dab = np.asarray(ctx_tca["E"]["LOAD"], float)
ci_fin_dab = np.asarray(ctx_tca["C_IMP_COP"]["T0"], float) / ctx_tca["TRM"]
ci_dab = C.senal_ajustada(ci_fin_dab, esc_dab)
ce_dab = np.asarray(ctx_tca["C_EXP_COP"], float) / ctx_tca["TRM"]
disp_dab = DT.alinear(ctx_tca["disp_v2"].to_numpy() if con_disp_dab else None, ctx_tca["fechas"])
x_dab = np.minimum(pv_dab, load_dab)
c_dab = dict(
    n="V2_p0", s="T0", esc=esc_dab, P=P_dab, b=b_dab, pe=pe_dab, f=ctx_tca["fechas"],
    load=load_dab, pv=pv_dab, ci=ci_dab, ci_fin=ci_fin_dab, ce=ce_dab, disp=disp_dab,
    delta=load_dab - x_dab, eps=pv_dab - x_dab, cv=np.asarray(P_dab["cv"], float),
    cv_fin=np.asarray(P_dab["cv_financiero"], float), pmin=np.asarray(P_dab["pmin"], float),
    pmax=np.asarray(P_dab["pmax"], float), cargos=np.broadcast_to(100.0, (4,)).copy(),
    intensidad=C.intensidad_termica_t_MWh(P_dab, esc_dab), fe_red=esc_dab.fe_red_t_MWh,
    tray=TRAYECTORIAS_V2_VIVO["V2_p0", "T0", "B_INT"],
)
for fila_ref in VENT_DAB[(VENT_DAB.Escenario == "V2_p0") & (VENT_DAB.Senal == "T0") & (VENT_DAB.Clase == "N4")].itertuples():
    j_idx = ["G1", "G2", "G3", "G4"].index(fila_ref.Unidad)
    modo_alt = "ALT1" if fila_ref.Alternativa.startswith("ALT1") else "ALT2"
    res_v = DAB.ventana(c_dab, int(fila_ref.Dia), j_idx, modo_alt, int(fila_ref.h_s))
    assert abs(res_v["Delta_ajustado"] - float(fila_ref.Delta_ajustado)) < 1e-6
    assert abs(res_v["Delta_emisiones"] - float(fila_ref.Delta_emisiones)) < 1e-6

print("Frecuencia anual de situaciones candidatas N1–N4 en el año 1 de K3:")
mostrar_tabla(FREQ_DAB, dec=1)

vista_vent = VENT_DAB[[
    "Escenario", "Senal", "Clase", "Fecha", "Unidad", "Alternativa",
    "h_s", "Delta_descarga_MWh", "Delta_ajustado", "Delta_financiero",
    "Delta_emisiones", "Delta_arranques", "Frontera_comun"
]].copy()
vista_vent.columns = [
    "Escenario", "Señal", "Clase", "Fecha", "Unidad", "Alternativa",
    "h_s", "ΔDescarga [MWh]", "ΔCosto Ajust. [USD]", "ΔCosto Fin. [USD]",
    "ΔEmisiones [t]", "ΔArranques", "Frontera común"
]
print("\nEfecto verificado de las alternativas ALT1 (apagado forzado post-plan) y ALT2 en las 6 ventanas:")
mostrar_tabla(vista_vent, dec=2)

Frecuencia anual de situaciones candidatas N1–N4 en el año 1 de K3:


| Escenario | Senal | N1_unidad_heredada | N2_cerca_del_minimo | N3_con_BESS_disponible | N4_sustitucion_sin_reasignar | N4_y_BESS_satura_ese_dia | Dias_distintos_N3 | Energia_mediana_N3_MWh | Unidades |
|---|---|---:|---:|---:|---:|---:|---:|---:|---|
| V2_p0 | T0 | 350 | 200 | 200 | 56 | 45 | 200 | 14,4 | G1,G2 |
| V2_p0 | T1 | 98 | 40 | 38 | 11 | 9 | 38 | 13,2 | G1,G2,G3 |
| V2_p10 | T0 | 294 | 167 | 167 | 24 | 20 | 167 | 12,0 | G1,G2 |
| V2_p10 | T1 | 83 | 35 | 35 | 10 | 9 | 35 | 14,4 | G1,G2 |
| V2_p20 | T1 | 70 | 31 | 31 | 7 | 6 | 31 | 12,0 | G2,G3 |
| V2_p50 | T1 | 50 | 21 | 21 | 7 | 7 | 21 | 12,0 | G2,G3 |


Efecto verificado de las alternativas ALT1 (apagado forzado post-plan) y ALT2 en las 6 ventanas:


| Escenario | Señal | Clase | Fecha | Unidad | Alternativa | h_s | ΔDescarga [MWh] | ΔCosto Ajust. [USD] | ΔCosto Fin. [USD] | ΔEmisiones [t] | ΔArranques | Frontera común |
|---|---|---|---|---|---|---:|---:|---:|---:|---:|---:|---:|
| V2_p0 | T0 | N4 | 2025-06-14 | G2 | ALT1_apagado_forzado_mejor_hora | 0 | 16,80 | -3.187,43 | -3.187,43 | -9,49 | 0 | True |
| V2_p0 | T0 | N4 | 2025-06-14 | G2 | ALT2_candidato_heredado_en_B | 0 | 0,00 | 0,00 | 0,00 | 0,00 | 0 | True |
| V2_p0 | T0 | N3_sin_N4 | 2025-12-15 | G2 | ALT1_apagado_forzado_mejor_hora | 7 | 0,00 | 41,87 | 41,87 | -2,16 | 0 | True |
| V2_p0 | T0 | N3_sin_N4 | 2025-12-15 | G2 | ALT2_candidato_heredado_en_B | 0 | 0,00 | 0,00 | 0,00 | 0,00 | 0 | True |
| V2_p0 | T1 | N4 | 2025-01-12 | G2 | ALT1_apagado_forzado_mejor_hora | 0 | 16,80 | -2.278,13 | -2.278,13 | -9,49 | 0 | True |
| V2_p0 | T1 | N4 | 2025-01-12 | G2 | ALT2_candidato_heredado_en_B | 0 | 0,00 | 0,00 | 0,00 | 0,00 | 0 | True |
| V2_p0 | T1 | N3_sin_N4 | 2025-12-28 | G2 | ALT1_apagado_forzado_mejor_hora | 0 | 0,00 | 100,00 | 100,00 | 0,00 | 1 | True |
| V2_p0 | T1 | N3_sin_N4 | 2025-12-28 | G2 | ALT2_candidato_heredado_en_B | 0 | 0,00 | 0,00 | 0,00 | 0,00 | 0 | True |
| V2_p10 | T0 | N4 | 2025-02-24 | G2 | ALT1_apagado_forzado_mejor_hora | 0 | 16,80 | -2.578,06 | -2.483,18 | -9,49 | 0 | True |
| V2_p10 | T0 | N4 | 2025-02-24 | G2 | ALT2_candidato_heredado_en_B | 0 | 0,00 | 0,00 | 0,00 | 0,00 | 0 | True |
| V2_p20 | T1 | N4 | 2025-08-30 | G2 | ALT1_apagado_forzado_mejor_hora | 0 | 12,00 | -1.628,68 | -1.493,14 | -6,78 | 0 | True |
| V2_p20 | T1 | N4 | 2025-08-30 | G2 | ALT2_candidato_heredado_en_B | 0 | 0,00 | 0,00 | 0,00 | 0,00 | 0 | True |

## 10 · Malla de dimensionamiento integrado `M01–M12` (6.400 anualidades, 600 VPN) y benchmarks finalistas

Para evaluar cómo interactúan el tamaño del campo solar, la potencia y energía del BESS, la señal tarifaria y el precio interno del carbono con la flota térmica existente fija, se resolvió la malla exploratoria `M01–M12`: **12 configuraciones $\times$ 4 precios de carbono $[0, 10, 20, 50\text{ USD/t}]$ $\times$ 2 señales $\times$ 3 reglas $\times$ 20 anualidades explícitas = 5.760 anualidades con BESS**, más **640 anualidades de cuatro referencias por contexto** (red + térmica y tres niveles de PV sin BESS), para **6.400 anualidades**. Las tablas financieras contienen **600 evaluaciones**: 576 del proyecto completo e incremental del BESS, más 24 de PV sin BESS; cada fila incluye VPN financiero y ajustado.

### 10.1 · Las tres comparaciones rigurosas (sin mezclar operación y tamaño)

1. **Comparación A (Reglas a tamaño fijo, `reglas_tamano_comun.csv`):**
   - `A_INT` gana en 6 de 8 escenarios en las configuraciones de mayor energía con alta potencia (`M01`, `M02`, `M05`), mientras `B_INT` gana los 2 restantes (`T0` con 20 y 50 USD/t, donde la térmica ya no opera).
   - `B_INT` predomina en `M06` y `M09` (6 de 8) y gana los 8 escenarios en `M11` ($57,2\text{ MWp}$ PV con BESS de $45\text{ MW} / 280\text{ MWh}$).
   - `D_INT` predomina en las baterías de menor energía (`M03`, `M04`, `M07`, `M08`, `M10`, `M12`).
2. **Comparación B (Mejor tamaño para cada regla, `mejores_por_regla.csv`):**
   - Para `A_INT`, el mejor tamaño de la malla es siempre **`M01` (`K3`: $114,4\text{ MWp}$, $45\text{ MW} / 280\text{ MWh}$)**.
   - Bajo `T1`, `B_INT` prefiere **`M07` ($25\text{ MW} / 180\text{ MWh}$) solo con carbono 0 USD/t**; `D_INT` la prefiere con **0 y 10 USD/t**. En los demás contextos ambas reglas eligen **`M01`**.
3. **Comparación C (Configuraciones distintas, `comparacion_configuraciones.csv`):**
   - Representando cada configuración por su mejor regla, **`M01` (`K3`) ocupa el primer puesto en los 8 escenarios evaluados**, superando a la segunda configuración distinta (`M05` o `M07`) entre $0,55$ y $1,50\text{ MUSD}$ ($0,80\text{–}1,57\,\%$ del VPN de la primera).

In [11]:
RES_MALLA = pd.read_csv(ROOT / "resultados_carbono_v2/malla_20260926/resultados_por_configuracion.csv")
COMP_CFG = pd.read_csv(ROOT / "resultados_carbono_v2/malla_20260926/comparacion_configuraciones.csv")
MEJ_REGLA = pd.read_csv(ROOT / "resultados_carbono_v2/malla_20260926/mejores_por_regla.csv")
REG_COMUN = pd.read_csv(ROOT / "resultados_carbono_v2/malla_20260926/reglas_tamano_comun.csv")
BM_FIN = pd.read_csv(ROOT / "resultados_carbono_v2/malla_20260926/benchmarks_finalistas.csv")

# Tabla de las 12 configuraciones M01-M12
cfg_malla = RES_MALLA.groupby("Config").agg(
    PV_MWp=("PV_MWp", "first"),
    P_MW=("P_MW", "first"),
    E_MWh=("E_MWh", "first"),
    Duracion_h=("Duracion_h", "first"),
    CAPEX_MUSD=("CAPEX_MUSD", "first"),
    OPEX_anual_MUSD=("OPEX_anual_MUSD", "first"),
).reset_index()

# Conteo de victorias por regla a tamaño fijo en los 8 escenarios (2 señales × 4 precios CO2)
vict_por_cfg = REG_COMUN.pivot_table(index="Config", columns="Mejor_regla", values="Senal", aggfunc="count", fill_value=0)
for col_r in ("A_INT", "B_INT", "D_INT"):
    if col_r not in vict_por_cfg.columns:
        vict_por_cfg[col_r] = 0
cfg_malla = cfg_malla.merge(vict_por_cfg[["A_INT", "B_INT", "D_INT"]], on="Config")
cfg_malla.columns = [
    "Config", "PV [MWp]", "P BESS [MW]", "E BESS [MWh]", "Duración [h]",
    "CAPEX Total [MUSD]", "OPEX [MUSD/año]", "Victorias A_INT (/8)", "Victorias B_INT (/8)", "Victorias D_INT (/8)"
]
print("Diseño de la malla M01–M12 y victorias por regla a tamaño fijo (Comparación A):")
mostrar_tabla(cfg_malla, dec=2)

# Tabla de Comparación C: Primera vs Segunda configuración distinta
comp_c_vista = COMP_CFG[[
    "Senal", "Precio_CO2_USD_t", "Config_1", "Regla_1", "VPN_ajustado_1_MUSD", "VPN_financiero_1_MUSD",
    "Config_2", "Regla_2", "VPN_ajustado_2_MUSD", "Diferencia_ajustado_MUSD"
]].copy()
comp_c_vista.columns = [
    "Señal", "CO₂ [USD/t]", "1ª Config", "Regla 1ª", "VPN Ajust. 1ª [MUSD]", "VPN Fin. 1ª [MUSD]",
    "2ª Config distinta", "Regla 2ª", "VPN Ajust. 2ª [MUSD]", "Ventaja 1ª − 2ª [MUSD]"
]
print("\nRanking de configuraciones distintas (Comparación C: M01 lidera en los 8 escenarios):")
mostrar_tabla(comp_c_vista, dec=3)

# Tabla de loscontrastes anuales contra los 17 benchmarks de los finalistas
bm_fin_vista = BM_FIN[BM_FIN.Regla_es_finalista].copy()
bm_fin_vista["Distancia al óptimo [%]"] = [
    intervalo_es(r.Diferencia_vs_factible_pct, r.Diferencia_vs_cota_pct, 2)
    for r in bm_fin_vista.itertuples()
]
bm_fin_vista["Costo Regla [MUSD]"] = bm_fin_vista["Costo_regla_ajustado_USD"] / 1e6
bm_fin_vista["Mejor Factible Benchmark [MUSD]"] = bm_fin_vista["Mejor_factible_USD"] / 1e6
bm_fin_vista["Cota Benchmark [MUSD]"] = bm_fin_vista["Cota_USD"] / 1e6
bm_fin_vista = bm_fin_vista[[
    "Config", "Senal", "Escenario_piloto", "Regla", "Benchmark",
    "Costo Regla [MUSD]", "Mejor Factible Benchmark [MUSD]", "Cota Benchmark [MUSD]",
    "Distancia al óptimo [%]", "Certificado_brecha_1e6"
]]
print("\nContraste operativo anual (año 1) de las configuraciones finalistas frente a sus benchmarks MILP:")
mostrar_tabla(bm_fin_vista, dec=4)

# Figura 15_7: Mapa de calor divergente del VPN incremental del BESS (izquierda) y efecto de E y PV (derecha)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.6, 5.2), layout="constrained")

# Mapa de calor del mejor VPN incremental ajustado del BESS por configuración y escenario
inc_piv = RES_MALLA.groupby(["Config", "Senal", "Precio_CO2_USD_t"])["VPN_incremental_ajustado_MUSD"].max().unstack(["Senal", "Precio_CO2_USD_t"])
cols_orden = [("T0", 0), ("T0", 10), ("T0", 20), ("T0", 50), ("T1", 0), ("T1", 10), ("T1", 20), ("T1", 50)]
inc_piv = inc_piv[cols_orden]
mat_vals = inc_piv.to_numpy()
vmax = float(np.abs(mat_vals).max())
im = ax1.imshow(mat_vals, cmap="RdYlGn", vmin=-vmax, vmax=vmax, aspect="auto")
for i in range(mat_vals.shape[0]):
    for j in range(mat_vals.shape[1]):
        v = mat_vals[i, j]
        ax1.text(j, i, es(v, 1), ha="center", va="center", fontsize=7.5,
                 color="white" if abs(v) > 22 else "#111111")
ax1.set_xticks(range(len(cols_orden)))
ax1.set_xticklabels([f"{s}\n{p} USD/t" for s, p in cols_orden], fontsize=8)
ax1.set_yticks(range(len(inc_piv.index)))
ax1.set_yticklabels([
    f"{c} ({int(cfg_malla.loc[i, 'PV [MWp]'])} MWp · {int(cfg_malla.loc[i, 'P BESS [MW]'])}/{int(cfg_malla.loc[i, 'E BESS [MWh]'])})"
    for i, c in enumerate(inc_piv.index)
], fontsize=8)
ax1.grid(False)
ax1.set_title("a) VPN incremental ajustado del BESS [MUSD] (rampa divergente centrada en 0)", loc="left", fontweight="bold", fontsize=9.5)
fig.colorbar(im, ax=ax1, shrink=0.82, label="VPN incremental BESS [MUSD]")

# Panel derecho: Curva de energía del BESS (120, 180, 280, 360 MWh con 45 MW y PV máximo: M04, M03, M01, M02)
cfgs_energia = ["M04", "M03", "M01", "M02"]
energias_x = [120, 180, 280, 360]
sub_e = RES_MALLA[RES_MALLA.Config.isin(cfgs_energia)].set_index(["Config", "Senal", "Precio_CO2_USD_t", "Regla"])
for (senal, p_co2, ls, marker) in [("T0", 0, "-", "o"), ("T0", 50, "--", "^"), ("T1", 0, "-", "s"), ("T1", 50, "--", "d")]:
    ys = [
        max(float(sub_e.loc[(c, senal, p_co2, r), "VPN_incremental_ajustado_MUSD"]) for r in ("A_INT", "B_INT", "D_INT"))
        for c in cfgs_energia
    ]
    color_l = "#7048c0" if senal == "T0" else "#c07a00"
    ax2.plot(energias_x, ys, color=color_l, ls=ls, marker=marker, lw=1.9, label=f"{senal} · {p_co2} USD/t CO₂")
    for x_val, y_val in zip(energias_x, ys):
        if x_val == 280:
            ax2.annotate(f"M01: {es(y_val, 1)}", (x_val, y_val), xytext=(0, 6), textcoords="offset points",
                         ha="center", fontsize=8, fontweight="bold")
ax2.axvline(280, color="#2a78d6", lw=0.9, ls=":", label="Óptimo interior (280 MWh, M01)")
ax2.set_xticks(energias_x)
ax2.set_xticklabels(["120 MWh\n(M04)", "180 MWh\n(M03)", "280 MWh\n(M01 / K3)", "360 MWh\n(M02)"])
ax2.set_xlabel("Capacidad energética del BESS (con P = 45 MW y PV = 114,4 MWp)")
ax2.set_ylabel("VPN incremental ajustado del BESS [MUSD]")
ax2.set_ylim(7.5, 36.0)
ax2.set_title("b) Óptimo interior en energía (280 MWh supera a 180 y 360 MWh)", loc="left", fontweight="bold", fontsize=9.5)
ax2.legend(frameon=False, loc="lower right", fontsize=8)

guardar_fig(fig, "15_7_etapa3_malla_dimensionamiento")

Diseño de la malla M01–M12 y victorias por regla a tamaño fijo (Comparación A):


| Config | PV [MWp] | P BESS [MW] | E BESS [MWh] | Duración [h] | CAPEX Total [MUSD] | OPEX [MUSD/año] | Victorias A_INT (/8) | Victorias B_INT (/8) | Victorias D_INT (/8) |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| M01 | 114,42 | 45,00 | 280,00 | 6,22 | 127,40 | 2,30 | 6 | 2 | 0 |
| M02 | 114,42 | 45,00 | 360,00 | 8,00 | 137,96 | 2,45 | 6 | 2 | 0 |
| M03 | 114,42 | 45,00 | 180,00 | 4,00 | 114,19 | 2,10 | 1 | 0 | 7 |
| M04 | 114,42 | 45,00 | 120,00 | 2,67 | 106,27 | 1,98 | 0 | 0 | 8 |
| M05 | 114,42 | 35,00 | 280,00 | 8,00 | 124,68 | 2,26 | 6 | 2 | 0 |
| M06 | 114,42 | 25,00 | 280,00 | 11,20 | 121,96 | 2,21 | 2 | 6 | 0 |
| M07 | 114,42 | 25,00 | 180,00 | 7,20 | 108,75 | 2,02 | 1 | 1 | 6 |
| M08 | 114,42 | 25,00 | 120,00 | 4,80 | 100,83 | 1,90 | 0 | 0 | 8 |
| M09 | 85,81 | 45,00 | 280,00 | 6,22 | 109,33 | 1,93 | 2 | 6 | 0 |
| M10 | 85,81 | 25,00 | 180,00 | 7,20 | 90,69 | 1,65 | 0 | 1 | 7 |
| M11 | 57,21 | 45,00 | 280,00 | 6,22 | 90,84 | 1,56 | 0 | 8 | 0 |
| M12 | 57,21 | 25,00 | 120,00 | 4,80 | 64,27 | 1,17 | 0 | 0 | 8 |


Ranking de configuraciones distintas (Comparación C: M01 lidera en los 8 escenarios):


| Señal | CO₂ [USD/t] | 1ª Config | Regla 1ª | VPN Ajust. 1ª [MUSD] | VPN Fin. 1ª [MUSD] | 2ª Config distinta | Regla 2ª | VPN Ajust. 2ª [MUSD] | Ventaja 1ª − 2ª [MUSD] |
|---|---:|---|---|---:|---:|---|---|---:|---:|
| T0 | 0 | M01 | A_INT | 83,365 | 83,365 | M05 | A_INT | 82,069 | 1,296 |
| T1 | 0 | M01 | A_INT | 68,270 | 68,270 | M07 | D_INT | 67,723 | 0,547 |
| T0 | 10 | M01 | A_INT | 88,688 | 83,151 | M05 | A_INT | 87,295 | 1,393 |
| T1 | 10 | M01 | A_INT | 71,259 | 68,333 | M05 | A_INT | 70,516 | 0,742 |
| T0 | 20 | M01 | B_INT | 93,599 | 91,509 | M05 | B_INT | 92,162 | 1,437 |
| T1 | 20 | M01 | A_INT | 74,109 | 68,555 | M05 | A_INT | 73,331 | 0,778 |
| T0 | 50 | M01 | B_INT | 96,735 | 91,509 | M05 | B_INT | 95,231 | 1,504 |
| T1 | 50 | M01 | A_INT | 81,679 | 70,113 | M05 | A_INT | 80,814 | 0,865 |


Contraste operativo anual (año 1) de las configuraciones finalistas frente a sus benchmarks MILP:


| Config | Senal | Escenario_piloto | Regla | Benchmark | Costo Regla [MUSD] | Mejor Factible Benchmark [MUSD] | Cota Benchmark [MUSD] | Distancia al óptimo [%] | Certificado_brecha_1e6 |
|---|---|---|---|---|---:|---:|---:|---|---:|
| M01 | T0 | V2_p0 | A_INT | OPT_AB | 4,7417 | 4,5964 | 4,5941 | [3,16; 3,22] | False |
| M01 | T0 | V2_p0 | B_INT | OPT_AB | 4,9860 | 4,5964 | 4,5941 | [8,47; 8,54] | False |
| M01 | T0 | V2_p0 | D_INT | OPT_D | 5,1183 | 4,5715 | 4,5695 | [11,96; 12,02] | False |
| M01 | T0 | V2_p10 | A_INT | OPT_AB | 4,8710 | 4,7791 | 4,7779 | [1,92; 1,95] | False |
| M01 | T0 | V2_p10 | B_INT | OPT_AB | 5,0946 | 4,7791 | 4,7779 | [6,60; 6,63] | False |
| M01 | T0 | V2_p10 | D_INT | OPT_D | 5,2618 | 4,7564 | 4,7550 | [10,62; 10,66] | False |
| M01 | T0 | V2_p20 | A_INT | OPT_AB | 4,9156 | 4,8505 | 4,8505 | [1,34; 1,35] | True |
| M01 | T0 | V2_p20 | B_INT | OPT_AB | 4,9058 | 4,8505 | 4,8505 | [1,13; 1,14] | True |
| M01 | T0 | V2_p20 | D_INT | OPT_D | 4,9111 | 4,8305 | 4,8305 | [1,67; 1,68] | True |
| M01 | T0 | V2_p50 | A_INT | OPT_AB | 5,0046 | 4,9454 | 4,9454 | [1,19; 1,20] | True |
| M01 | T0 | V2_p50 | B_INT | OPT_AB | 4,9948 | 4,9454 | 4,9454 | [0,99; 1,00] | True |
| M01 | T0 | V2_p50 | D_INT | OPT_D | 5,0012 | 4,9278 | 4,9278 | [1,48; 1,49] | True |
| M01 | T1 | V2_p0 | A_INT | OPT_AB | 4,0441 | 3,8530 | 3,8527 | [4,96; 4,97] | False |
| M01 | T1 | V2_p10 | A_INT | OPT_AB | 4,1033 | 3,9274 | 3,9272 | [4,48; 4,49] | False |
| M01 | T1 | V2_p10 | B_INT | OPT_AB | 4,2135 | 3,9274 | 3,9272 | [7,28; 7,30] | False |
| M01 | T1 | V2_p20 | A_INT | OPT_AB | 4,1600 | 3,9939 | 3,9938 | [4,15; 4,17] | False |
| M01 | T1 | V2_p20 | B_INT | OPT_AB | 4,2574 | 3,9939 | 3,9938 | [6,59; 6,61] | False |
| M01 | T1 | V2_p20 | D_INT | OPT_D | 4,3060 | 3,9809 | 3,9808 | [8,16; 8,17] | False |
| M01 | T1 | V2_p50 | A_INT | OPT_AB | 4,3111 | 4,1594 | 4,1594 | [3,64; 3,65] | True |
| M01 | T1 | V2_p50 | B_INT | OPT_AB | 4,4020 | 4,1594 | 4,1594 | [5,83; 5,84] | True |
| M01 | T1 | V2_p50 | D_INT | OPT_D | 4,4393 | 4,1518 | 4,1518 | [6,92; 6,93] | True |
| M07 | T1 | V2_p0 | B_INT | OPT_AB | 6,6924 | 6,5498 | 6,5479 | [2,17; 2,21] | False |
| M07 | T1 | V2_p0 | D_INT | OPT_D | 6,6910 | 6,5443 | 6,5437 | [2,24; 2,26] | False |
| M07 | T1 | V2_p10 | D_INT | OPT_D | 6,8101 | 6,6632 | 6,6626 | [2,20; 2,22] | False |

## 11 · Sensibilidad financiera al segmento de costo local del BESS (`H_B4` frente a `H_B3`) — Ejecución en vivo

En `CurvasCapexPVBESS.xlsx`, todas las configuraciones de la malla (`120–360 MWh`) caen en el segmento **`B4` ($E > 100\text{ MWh}$)**, cuyo costo local por potencia es $203,4\text{ USD/kW}$ y su costo fijo es $900.000\text{ USD}$, frente a $86,6\text{ USD/kW}$ y $526.263\text{ USD}$ en el segmento **`B3` ($20\text{–}100\text{ MWh}$)**. Como la documentación disponible no asocia el umbral de $100\text{ MWh}$ a un cambio físico de tensión verificado para la planta, se evalúa qué ocurre si las 12 configuraciones se costean con los parámetros locales de **`B3` (`H_B3`)** frente al caso base **`B4` (`H_B4`)**:

- **Diferencia de CAPEX a igual energía y potencia (por ejemplo en $100\text{ MWh}$):** la diferencia `B4 − B3` depende únicamente de la potencia del BESS y es de **$13,77\text{ mil millones COP}$ ($4,405\text{ MUSD}$) con $25\text{ MW}$**, **$18,65\text{ mil millones COP}$ ($5,967\text{ MUSD}$) con $35\text{ MW}$** y **$23,53\text{ mil millones COP}$ ($7,528\text{ MUSD}$) con $45\text{ MW}$** (distinguiéndose del salto de la curva entre $100$ y $100,1\text{ MWh}$, que incluye además $0,1\text{ MWh}$ de energía adicional).
- **Efecto sobre el VPN ($\Delta\text{VPN}_{\text{B3}-\text{B4}}$):** al sumar el menor CAPEX y el menor OPEX fijo anual ($1,5\,\%$ del CAPEX descontado con el factor de anualidad de 20 años al $10\,\%$, $\text{FA} = 8,513564$), el VPN mejora en **$+8,489843\text{ MUSD}$ ($26,54\text{ mil MM COP}$) para $45\text{ MW}$**, **$+6,728458\text{ MUSD}$ ($21,03\text{ mil MM COP}$) para $35\text{ MW}$** y **$+4,967072\text{ MUSD}$ ($15,53\text{ mil MM COP}$) para $25\text{ MW}$**.

La celda siguiente **recalcula en vivo** desde el libro Excel los costos y el incremento analítico de VPN para todas las configuraciones, comprobando su coincidencia exacta con `resultados_carbono_v2/sensibilidad_capex_bess_20260927/`.

In [12]:
import sensibilidad_capex_bess as SCB

CAPEX_B4_B3 = pd.read_csv(ROOT / "resultados_carbono_v2/sensibilidad_capex_bess_20260927/capex_opex_por_configuracion.csv")
ELEC_CFG_B4_B3 = pd.read_csv(ROOT / "resultados_carbono_v2/sensibilidad_capex_bess_20260927/eleccion_configuraciones_B4_B3.csv")
ELEC_TAM_B4_B3 = pd.read_csv(ROOT / "resultados_carbono_v2/sensibilidad_capex_bess_20260927/eleccion_tamano_por_regla_B4_B3.csv")

# Recálculo en vivo de CAPEX, OPEX y ΔVPN con sensibilidad_capex_bess.py
par_libro, prima_libro = SCB.parametros_libro()
FA_20_10 = sum(1.0 / (1.0 + costos.WACC) ** y for y in range(1, 21))
filas_capex_vivo = []
for cfg_row in cfg_malla.itertuples():
    cid = cfg_row.Config
    p_mw = float(cfg_row._3)
    e_mwh = float(cfg_row._4)
    c_b4 = SCB.capex_bess_segmento_USD(p_mw, e_mwh, par_libro["B4"], prima_libro)
    c_b3 = SCB.capex_bess_segmento_USD(p_mw, e_mwh, par_libro["B3"], prima_libro)
    opex_b4 = 0.015 * c_b4
    opex_b3 = 0.015 * c_b3
    delta_capex_musd = (c_b4 - c_b3) / 1e6
    delta_vpn_musd = ((c_b4 - c_b3) + (opex_b4 - opex_b3) * FA_20_10) / 1e6

    ref_b4 = CAPEX_B4_B3[(CAPEX_B4_B3.Config == cid) & (CAPEX_B4_B3.Hipotesis == "H_B4")].iloc[0]
    ref_b3 = CAPEX_B4_B3[(CAPEX_B4_B3.Config == cid) & (CAPEX_B4_B3.Hipotesis == "H_B3")].iloc[0]
    assert abs(c_b4 / 1e6 - float(ref_b4.CAPEX_BESS_MUSD)) < 1e-9
    assert abs(c_b3 / 1e6 - float(ref_b3.CAPEX_BESS_MUSD)) < 1e-9

    filas_capex_vivo.append({
        "Config": cid,
        "P [MW]": p_mw,
        "E [MWh]": e_mwh,
        "CAPEX BESS H_B4 [MUSD]": c_b4 / 1e6,
        "CAPEX BESS H_B3 [MUSD]": c_b3 / 1e6,
        "B4 − B3 CAPEX [mil MM COP]": delta_capex_musd * costos.TRM_COP_USD / 1e3,
        "ΔVPN (H_B3 − H_B4) [MUSD]": delta_vpn_musd,
        "ΔVPN [mil MM COP]": delta_vpn_musd * costos.TRM_COP_USD / 1e3,
    })

DF_CAPEX_VIVO = pd.DataFrame(filas_capex_vivo)
print("Recálculo en vivo de CAPEX, OPEX y mejora de VPN entre las hipótesis H_B4 y H_B3:")
mostrar_tabla(DF_CAPEX_VIVO, dec=2)

# Casos donde cambia el mejor tamaño por regla entre H_B4 y H_B3
cambios_tam = ELEC_TAM_B4_B3[ELEC_TAM_B4_B3.Cambia_tamano][[
    "Senal", "Precio_CO2_USD_t", "Regla", "Config_B4", "VPN_ajustado_MUSD_B4",
    "Config_B3", "VPN_ajustado_MUSD_B3", "Segundo_config_B3", "Ventaja_sobre_segundo_MUSD_B3"
]].copy()
cambios_tam.columns = [
    "Señal", "CO₂ [USD/t]", "Regla", "Mejor H_B4", "VPN Ajust. H_B4 [MUSD]",
    "Mejor H_B3", "VPN Ajust. H_B3 [MUSD]", "2º en H_B3", "Ventaja sobre 2º [MUSD]"
]
print(f"\n{len(cambios_tam)} contextos de regla, señal y carbono (de {len(ELEC_TAM_B4_B3)}) cambian de M07 (25 MW/180 MWh) a M01 (45 MW/280 MWh) bajo H_B3:")
mostrar_tabla(cambios_tam, dec=3)

# Figura 15_8: Comparación de VPN ajustado por configuración en H_B4 vs H_B3 (T0 y T1 a 0 USD/t)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.6, 4.6), sharey=True, layout="constrained")
cfgs_lista = list(cfg_malla["Config"])
x_m = np.arange(len(cfgs_lista))
w_m = 0.36

for ax, senal, titulo_s in [(ax1, "T0", "a) Señal T0 (0 USD/t CO₂): M01 gana en H_B4 y H_B3"),
                            (ax2, "T1", "b) Señal T1 (0 USD/t CO₂): en H_B3 la 2ª pasa de M07 a M05")]:
    sub_0 = RES_MALLA[(RES_MALLA.Senal == senal) & (RES_MALLA.Precio_CO2_USD_t == 0)]
    mejor_b4 = sub_0.groupby("Config")["VPN_ajustado_MUSD"].max().loc[cfgs_lista].to_numpy()
    delta_v = DF_CAPEX_VIVO.set_index("Config").loc[cfgs_lista, "ΔVPN (H_B3 − H_B4) [MUSD]"].to_numpy()
    mejor_b3 = mejor_b4 + delta_v

    ax.bar(x_m - w_m/2, mejor_b4, width=w_m, color="#2a78d6", label="H_B4 (segmento base > 100 MWh)")
    ax.bar(x_m + w_m/2, mejor_b3, width=w_m, color="#1baf7a", label="H_B3 (sensibilidad costo local)")
    ax.scatter([0 - w_m/2, 0 + w_m/2], [mejor_b4[0] + 2.2, mejor_b3[0] + 2.2], color="#eb6834", marker="*", s=95, zorder=4, label="Mejor tamaño evaluado (M01)")
    ax.set_xticks(x_m)
    ax.set_xticklabels(cfgs_lista, rotation=35)
    ax.set_ylim(0, 108)
    ax.set_title(titulo_s, loc="left", fontweight="bold", fontsize=9.5)
    ax.legend(frameon=False, loc="upper right", fontsize=8)
ax1.set_ylabel("Mayor VPN ajustado por configuración [MUSD]")

guardar_fig(fig, "15_8_etapa3_sensibilidad_b4_b3")

Recálculo en vivo de CAPEX, OPEX y mejora de VPN entre las hipótesis H_B4 y H_B3:


| Config | P [MW] | E [MWh] | CAPEX BESS H_B4 [MUSD] | CAPEX BESS H_B3 [MUSD] | B4 − B3 CAPEX [mil MM COP] | ΔVPN (H_B3 − H_B4) [MUSD] | ΔVPN [mil MM COP] |
|---|---:|---:|---:|---:|---:|---:|---:|
| M01 | 45,00 | 280,00 | 50,42 | 42,89 | 23,53 | 8,49 | 26,54 |
| M02 | 45,00 | 360,00 | 60,98 | 53,45 | 23,53 | 8,49 | 26,54 |
| M03 | 45,00 | 180,00 | 37,21 | 29,68 | 23,53 | 8,49 | 26,54 |
| M04 | 45,00 | 120,00 | 29,29 | 21,76 | 23,53 | 8,49 | 26,54 |
| M05 | 35,00 | 280,00 | 47,70 | 41,73 | 18,65 | 6,73 | 21,03 |
| M06 | 25,00 | 280,00 | 44,98 | 40,57 | 13,77 | 4,97 | 15,53 |
| M07 | 25,00 | 180,00 | 31,77 | 27,37 | 13,77 | 4,97 | 15,53 |
| M08 | 25,00 | 120,00 | 23,85 | 19,45 | 13,77 | 4,97 | 15,53 |
| M09 | 45,00 | 280,00 | 50,42 | 42,89 | 23,53 | 8,49 | 26,54 |
| M10 | 25,00 | 180,00 | 31,77 | 27,37 | 13,77 | 4,97 | 15,53 |
| M11 | 45,00 | 280,00 | 50,42 | 42,89 | 23,53 | 8,49 | 26,54 |
| M12 | 25,00 | 120,00 | 23,85 | 19,45 | 13,77 | 4,97 | 15,53 |


3 contextos de regla, señal y carbono (de 24) cambian de M07 (25 MW/180 MWh) a M01 (45 MW/280 MWh) bajo H_B3:


| Señal | CO₂ [USD/t] | Regla | Mejor H_B4 | VPN Ajust. H_B4 [MUSD] | Mejor H_B3 | VPN Ajust. H_B3 [MUSD] | 2º en H_B3 | Ventaja sobre 2º [MUSD] |
|---|---:|---|---|---:|---|---:|---|---:|
| T1 | 0 | B_INT | M07 | 67,712 | M01 | 76,002 | M05 | 2,312 |
| T1 | 0 | D_INT | M07 | 67,723 | M01 | 75,533 | M05 | 2,234 |
| T1 | 10 | D_INT | M07 | 70,220 | M01 | 78,475 | M05 | 2,284 |

---

## 12 · Síntesis General de Conclusiones, Alcance y Verificación de Integridad

### 12.1 · Cuadro sinóptico de hallazgos por etapa

| Etapa del estudio | Pregunta central | Hallazgo cuantitativo verificado |
|---|---|---|
| **Etapa I**<br>*(PV + BESS + Red,*<br>*sin térmica)* | ¿Qué determina el valor del BESS y cuánto pierden las heurísticas frente a previsión perfecta? | • Las reglas orientadas al usuario (`A`, `B`, `D`) siempre generan $\Delta\text{VPN}_{\text{BESS}} > 0$ ($+14,71\text{ a }+43,92\text{ MUSD}$), `C` y `F` concentran los **46 casos negativos** del diseño completo, pero no todos sus casos destruyen valor.<br>• Restringido a `A/B/D`, la configuración `K` explica el **$66,38\,\%$** de la variación de $\Delta\text{VPN}_{\text{BESS}}$, la señal el **$28,02\,\%$**, el sitio el **$4,11\,\%$** y la elección entre `A`, `B` y `D` apenas el **$0,07\,\%$**.<br>• Frente a previsión perfecta, la **brecha de decisión** (`OPT_B − B`) es de solo **$0,35\text{–}1,43\,\%$**, mientras que la **brecha de flexibilidad** (`OPT_FULL − OPT_B`) es de **$2,05\text{–}3,79\,\%$** (entre 2,33 y 5,83 veces la brecha de decisión). |
| **Etapa II**<br>*(PV + BESS +*<br>*Térmica G1–G4 + Red)* | ¿Cómo interactúan las reglas operativas con una flota térmica con mínimos técnicos y costos de arranque? | • En **K3** ($114,4\text{ MWp}$; $45\text{ MW} / 280\text{ MWh}$), **`A_INT`** logra el menor costo operativo anual ($4,7396\text{ MUSD/año}$ en `T0` y $4,0439\text{ MUSD/año}$ en `T1`) y el mayor VPN a 20 años ($81,48\text{ y }60,14\text{ mil MM COP}$ de VPN incremental del BESS).<br>• En el **factorial térmico `K1–K4`**, `A_INT` gana cuando el almacenamiento es abundante (`K1`, `K3`) y **`D_INT` gana cuando el almacenamiento es más escaso (`K2`, `K4`)**, con ventajas de $0,40\text{–}1,19\,\%$.<br>• `C_INT` y `F_INT` coinciden bajo `T0`, pero difieren y sí cargan desde red bajo `T1`; conservan energía sin descargar en horas de déficit y tienen VPN incremental del BESS negativo ($-89,29\text{ mil MM COP}$ en `T0`; $-43,22$ para C y $-43,23\text{ mil MM COP}$ para F en `T1`). |
| **Etapa III**<br>*(Extensión V2:*<br>*disponibilidad, carbono*<br>*y dimensionamiento)* | ¿Cómo modifican la disponibilidad térmica y un precio interno de carbono el despacho, las emisiones y el tamaño preferido? | • En K3 bajo `T0`, los escenarios evaluados de **20 y 50 USD/t** eliminan la generación térmica. Con `A_INT`, las emisiones operativas acumuladas en 20 años pasan de **$291,6\text{ kt}$** en S0 a **$64,0\text{ kt}$** en esos escenarios. En el año 1, `A_INT/B_INT/D_INT` quedan a **$1,0\text{–}1,7\,\%$** de sus benchmarks certificados; es una distancia de costo operativo anual, no de VPN.<br>• En la malla de 12 configuraciones (`M01–M12`, 6.400 anualidades), **`M01` (`K3`) ocupa el primer puesto entre los tamaños evaluados en los 8 escenarios** tanto bajo la hipótesis de costo `H_B4` como bajo `H_B3`, con un óptimo interior en energía ($280\text{ MWh}$ supera a $180$ y $360\text{ MWh}$).<br>• Bajo `T1` con `H_B4`, `B_INT` prefiere `M07` ($25\text{ MW} / 180\text{ MWh}$) con carbono 0, y `D_INT` con 0 y 10 USD/t. Esos tres contextos cambian a `M01` con `H_B3`; los demás ya prefieren `M01`. |

### 12.2 · Verificación final de integridad y exportación de trazabilidad

La celda siguiente comprueba que ninguna de las tablas congeladas del repositorio cambió su huella SHA-256 durante la ejecución del cuaderno y guarda el manifiesto completo en `resultados_nb15/15_trazabilidad_ejecucion.json`.

In [13]:
# Comprobación estricta de que ningún archivo fuente fue modificado
for ruta_str, sha_ini in HASHES_INICIALES.items():
    sha_fin = hashlib.sha256(Path(ruta_str).read_bytes()).hexdigest()
    assert sha_fin == sha_ini, f"Archivo fuente alterado: {ruta_str}"

figuras_generadas = sorted(p.name for p in FIG.glob("15_*.*"))
assert len(figuras_generadas) == 16, f"Se esperaban 16 archivos de figura (8 PNG + 8 SVG), encontrados: {len(figuras_generadas)}"

manifiesto_nb15 = {
    "cuaderno": "15_Sintesis_integral_tesis_V2.ipynb",
    "rama": "tesis-v2-parametros",
    "numba_activo": NUMBA_ACTIVO,
    "simulaciones_en_vivo": {
        "etapa_1_multianual_20_anios_casos": len(DF_VIVO_E1) + 2,
        "etapa_2_anual_8760h_casos": len(DF_V1_VIVO),
        "etapa_2_demostracion_CF_72h": 2,
        "etapa_3_anual_carbono_8760h_casos": len(DF_V2_VIVO),
        "etapa_3_recalculo_capex_B4_B3_configs": len(DF_CAPEX_VIVO),
    },
    "figuras_generadas": figuras_generadas,
    "fuentes_sha256_verificadas": HASHES_INICIALES,
    "fuentes_intactas": True,
}
(OUT / "15_trazabilidad_ejecucion.json").write_text(
    json.dumps(manifiesto_nb15, ensure_ascii=False, indent=2), encoding="utf-8"
)
DF_VIVO_E1.to_csv(OUT / "15_etapa1_simulacion_vivo_K3.csv", index=False)
DF_V1_VIVO.to_csv(OUT / "15_etapa2_simulacion_vivo_K3_V1.csv", index=False)
DF_V2_VIVO.to_csv(OUT / "15_etapa3_simulacion_vivo_K3_V2.csv", index=False)
DF_CAPEX_VIVO.to_csv(OUT / "15_etapa3_sensibilidad_B4_B3_vivo.csv", index=False)

print("Verificación final completada con éxito:")
print(f"  • {len(HASHES_INICIALES)} fuentes maestras del repositorio verificadas por SHA-256 (0 alteraciones).")
print(f"  • {len(DF_VIVO_E1)+2} corridas multianuales (Etapa I) + {len(DF_V1_VIVO)} anuales (Etapa II) + {len(DF_V2_VIVO)} anuales con carbono (Etapa III) ejecutadas en vivo.")
print(f"  • {len(figuras_generadas)} archivos gráficos (8 PNG + 8 SVG) exportados en {FIG.relative_to(ROOT)}/.")
print(f"  • Tablas y trazabilidad guardadas en {OUT.relative_to(ROOT)}/15_trazabilidad_ejecucion.json.")

Verificación final completada con éxito:
  • 10 fuentes maestras del repositorio verificadas por SHA-256 (0 alteraciones).
  • 12 corridas multianuales (Etapa I) + 10 anuales (Etapa II) + 20 anuales con carbono (Etapa III) ejecutadas en vivo.
  • 16 archivos gráficos (8 PNG + 8 SVG) exportados en figuras_nb15/.
  • Tablas y trazabilidad guardadas en resultados_nb15/15_trazabilidad_ejecucion.json.
